# ROGII TVT v22 — **OOF ANALYSIS FORK** (do not submit)

This is v22 with cells **P0–P6** appended. It produces `v22_oof.pkl` for the
BiGRU join. It is **not** a submission notebook.

**Run All is safe.** The test-inference / submission cell is guarded by
`RUN_TEST_INFERENCE = False`, so it will be skipped. Leave it that way — after
P1–P3 run, `build_ufield`, `_field_query`, `field_blend` and `predict_well_diag`
are all modified in memory, and you do not want a scoring run touching them.

**What P0–P6 do to v22:** nothing destructive. P2 and P3 define new globals that
shadow the originals; P1 rewrites `build_ufield`'s source text via
`inspect.getsource`, asserting each of four anchors matches exactly once. Your
model-code cell is never edited.

**A note on `_attest`.** Function sources change after P1–P3, so the attestation
hash in this fork will differ from the sealed v22. That is expected here and is
precisely why this is a fork.

**Expected timeline:** v22 cells ~1 min, P4 ~40 s, P5 ~10 s, P6 45–75 min.

# ROGII TVT v22 — Corrupted-Gate Weight Retune (Attested)

Base = v19 (public LB 8.926). Single change: the drift-cancel and typewell-only branch
weights, retuned against a statistically-powered corrupted suite.

**Motivation (v20/v21 post-mortems):** v20 (cohort augmentation, 10.625) and v21 (fine
reference, 9.019) both failed by widening the hidden gap. v21's lesson: the corrupted
subpopulation is what the hidden test set punishes, and the old 8-well/2-shape gate was
too weak to steer by. We built a 20-well x 4-shape corrupted suite (ramp/sine/gain/step,
80 baselines) and retuned w_dc 0.30->0.40, w_tw 0.25->0.28.

**Evidence:** corrupted suite improves -0.35 MAE (paired-t p<0.001), consistent across
all four shapes (55/80 paired wins). Clean full-200: 6.577 vs v19 6.554 (neutral, a
wash). Pad-holdout: 7.357 (neutral). The bet is deliberate: identical on easy wells,
significantly more robust on the corrupted subpopulation that drives the gap.

Fine reference (v21) and cohort augmentation (v20) both reverted and marked LB-falsified
in CONFIG. Runtime ~3.6 s/well + ~40 s field build. Attestation prints below.


In [ ]:
import numpy as np
import pandas as pd
import glob, os, time

DATA = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'test' in dirs and glob.glob(os.path.join(root, 'test', '*__horizontal_well.csv')):
        DATA = root
        break
if DATA is None:
    DATA = next(p for p in ['data', '.'] if os.path.isdir(os.path.join(p, 'test')))
os.environ['ROGII_DATA'] = os.path.abspath(DATA)
print('data dir:', DATA)
print('test wells:', len(glob.glob(os.path.join(DATA, 'test', '*__horizontal_well.csv'))))

## Model code

In [ ]:
DATA = os.environ.get('ROGII_DATA', 'data')

# ------------------------------------------------------------------ 1. IO

_H_COLS = ['X', 'Y', 'MD', 'Z', 'GR', 'TVT_input', 'TVT']
def _read_csv(path, usecols=None):
    try:
        cols = pd.read_csv(path, nrows=0).columns
        use = [c for c in usecols if c in cols] if usecols else None
        try:
            return pd.read_csv(path, usecols=use, engine='pyarrow')
        except Exception:
            return pd.read_csv(path, usecols=use)
    except Exception:
        return pd.read_csv(path)

def load_well(split, well):
    h = _read_csv(f'{DATA}/{split}/{well}__horizontal_well.csv', _H_COLS)
    t = _read_csv(f'{DATA}/{split}/{well}__typewell.csv')
    h.attrs['well'] = well
    return h, t

def wells(split):
    return sorted(os.path.basename(f).split('__')[0]
                  for f in glob.glob(f'{DATA}/{split}/*__horizontal_well.csv'))

# ---------------------------------------------------- 2. signal utilities

def smooth(x, w):
    """Edge-padded moving average; w<=1 is a copy (never aliases input)."""
    x = np.asarray(x, dtype=float)
    if w <= 1 or len(x) < 2:
        return x.copy()
    w = min(int(w), len(x))
    k = np.ones(w) / w
    xp = np.pad(x, (w // 2, w - w // 2 - 1), mode='edge')
    return np.convolve(xp, k, mode='valid')

def interp_gaps(x, max_gap):
    """Interpolate interior NaN runs of <= max_gap samples; leave longer runs
    and lead/tail NaNs as NaN (extrapolating GR would fabricate signal)."""
    x = np.asarray(x, dtype=float).copy()
    isn = ~np.isfinite(x)
    if not isn.any() or isn.all():
        return x
    idx = np.arange(len(x))
    xi = np.interp(idx, idx[~isn], x[~isn])
    d = np.diff(np.concatenate(([0], isn.astype(np.int8), [0])))
    for s, e in zip(np.where(d == 1)[0], np.where(d == -1)[0]):
        if e - s > max_gap:
            xi[s:e] = np.nan
    return xi

def _fit_affine(x, y, min_pts=50, trim_q=0.8):
    """Robust-ish affine y ~ a*x + b (two-pass trimmed LS). Returns (a, b).
    Degenerate inputs (few points / zero variance) -> identity mapping."""
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < min_pts or np.std(x[m]) < 1e-9:
        return 1.0, 0.0
    a, b = np.polyfit(x[m], y[m], 1)
    r = np.abs(y[m] - (a * x[m] + b))
    keep = r < np.quantile(r, trim_q)
    if keep.sum() >= min_pts:
        a, b = np.polyfit(x[m][keep], y[m][keep], 1)
    return float(a), float(b)

# ------------------------------------------------- 3. validation & arrays

class GuardError(Exception):
    """Raised when a well violates a structural assumption; carries a status tag."""
    def __init__(self, status):
        self.status = status
        super().__init__(status)

def prepare_arrays(h, max_gap_ft=25.0):
    """Validate the horizontal frame and return a dict of clean arrays.

    Guards (raise GuardError):
      no_blind        - nothing to predict
      no_known        - TVT_input entirely NaN (no anchor exists)
      blind_at_start  - blind zone begins at row 0 (no anchor before it)
      bad_spacing     - MD spacing non-positive or wildly irregular
    """
    n = len(h)
    if n < 10:
        raise GuardError('too_short')
    tvt_in = h['TVT_input'].to_numpy(dtype=float)
    blind = ~np.isfinite(tvt_in)
    if not blind.any():
        raise GuardError('no_blind')
    if blind.all():
        raise GuardError('no_known')
    k0 = int(np.argmax(blind))
    if k0 == 0:
        raise GuardError('blind_at_start')

    md = h['MD'].to_numpy(dtype=float)
    dmd_all = np.diff(md)
    dmd = float(np.median(dmd_all)) if len(dmd_all) else 1.0
    if not np.isfinite(dmd) or dmd <= 0:
        raise GuardError('bad_spacing')

    z = h['Z'].to_numpy(dtype=float)
    if not np.isfinite(z).all():                    # sparse Z: interpolate on MD
        ok = np.isfinite(z)
        if ok.sum() < 2:
            raise GuardError('no_z')
        z = np.interp(md, md[ok], z[ok])

    gr_raw = h['GR'].to_numpy(dtype=float)
    gr = interp_gaps(gr_raw, max_gap=max(1, int(round(max_gap_ft / dmd))))
    anchor_tvt = float(tvt_in[k0 - 1])

    return dict(n=n, blind=blind, k0=k0, md=md, dmd=dmd, z=z, gr=gr,
                gr_raw=gr_raw, tvt_in=tvt_in, anchor_tvt=anchor_tvt)

# ------------------------------------------------------------ 4. reference

def build_reference(arr, t, grid_step=0.5, type_smooth_ft=1.0,
                    pseudo_smooth_ft=1.5, n0=3.0, band_pad=700.0,
                    extra_tvt=None, extra_gr=None, extra_w=0.5):
    """GR reference in horizontal-tool units on a TVT grid.

    Blend of (a) typewell affine-mapped into horizontal units and (b) a
    pseudo-typewell binned from the known zone's (TVT_input, GR) pairs.
    Blend weight w = smoothed_count / (smoothed_count + n0), forced to 0
    outside the pseudo's covered TVT range.
    """
    t = t.dropna(subset=['TVT', 'GR']).sort_values('TVT')
    tw_tvt = t['TVT'].to_numpy(dtype=float)
    tw_gr = t['GR'].to_numpy(dtype=float)
    if len(tw_tvt) < 20:
        raise GuardError('typewell_short')
    grid_t = np.arange(tw_tvt[0], tw_tvt[-1] + grid_step, grid_step)
    g_t = smooth(np.interp(grid_t, tw_tvt, tw_gr),
                 max(1, int(round(type_smooth_ft / grid_step))))

    known = np.isfinite(arr['tvt_in']) & np.isfinite(arr['gr_raw'])
    tvt_k = arr['tvt_in'][known]
    gr_k = arr['gr_raw'][known]

    # typewell -> horizontal units
    g_at_known = np.interp(tvt_k, grid_t, g_t, left=np.nan, right=np.nan)
    A, B = _fit_affine(g_at_known, gr_k)

    lo = min(grid_t[0], tvt_k.min() if len(tvt_k) else grid_t[0],
             arr['anchor_tvt'] - band_pad)
    hi = max(grid_t[-1], tvt_k.max() if len(tvt_k) else grid_t[-1],
             arr['anchor_tvt'] + band_pad)
    grid = np.arange(lo, hi + grid_step, grid_step)

    a_grid = np.full(len(grid), A)
    b_grid = np.full(len(grid), B)
    if CONFIG.get('formation_affine', False) and 'Geology' in t.columns:
        labs_tw = np.array([str(x) for x in t['Geology'].tolist()])
        valid_lab = np.array([s not in ('nan', 'None', '') for s in labs_tw])
        if valid_lab.sum() >= 5 and len(tvt_k) > 0:
            pos_k = np.clip(np.searchsorted(tw_tvt, tvt_k), 0, len(tw_tvt) - 1)
            lab_k = labs_tw[pos_k]
            pos_g = np.clip(np.searchsorted(tw_tvt, grid), 0, len(tw_tvt) - 1)
            lab_g = labs_tw[pos_g]
            in_tw = (grid >= tw_tvt[0]) & (grid <= tw_tvt[-1])
            mp = int(CONFIG.get('formation_min_pairs', 40))
            shr = float(CONFIG.get('formation_shrink', 60.0))
            for g_lab in np.unique(lab_k):
                if g_lab in ('nan', 'None', ''):
                    continue
                mk = (lab_k == g_lab) & np.isfinite(g_at_known)
                if mk.sum() < mp or np.std(g_at_known[mk]) < 6.0:
                    continue
                try:
                    a_f, b_f = _fit_affine(g_at_known[mk], gr_k[mk],
                                           min_pts=mp)
                except Exception:
                    continue
                if not (np.isfinite(a_f) and np.isfinite(b_f)
                        and 0.2 < a_f < 5.0):
                    continue
                lam = mk.sum() / (mk.sum() + shr)
                sel = (lab_g == g_lab) & in_tw
                a_grid[sel] = lam * a_f + (1 - lam) * A
                b_grid[sel] = lam * b_f + (1 - lam) * B
            fade = max(1, int(round(CONFIG.get('formation_fade_ft', 12.0)
                                    / grid_step)))
            a_grid = smooth(a_grid, fade)
            b_grid = smooth(b_grid, fade)
    ref = a_grid * np.interp(grid, grid_t, g_t) + b_grid

    if len(tvt_k) > 20:
        bins = np.clip(((tvt_k - grid[0]) / grid_step).astype(int), 0, len(grid) - 1)
        ssum = np.bincount(bins, weights=gr_k, minlength=len(grid))
        cnt = np.bincount(bins, minlength=len(grid)).astype(float)
        if extra_tvt is not None and len(extra_tvt):
            me = (np.isfinite(extra_tvt) & np.isfinite(extra_gr)
                  & (extra_tvt >= grid[0]) & (extra_tvt <= grid[-1]))
            be = ((extra_tvt[me] - grid[0]) / grid_step).astype(int)
            be = np.minimum(be, len(grid) - 1)
            ssum = ssum + np.bincount(be, weights=extra_gr[me] * extra_w,
                                      minlength=len(grid))
            cnt = cnt + extra_w * np.bincount(be, minlength=len(grid))
        cov = cnt > 0
        if cov.sum() > 20:
            idxg = np.arange(len(grid))
            pseudo = np.interp(idxg, idxg[cov], ssum[cov] / cnt[cov])
            pseudo = smooth(pseudo, max(1, int(round(pseudo_smooth_ft / grid_step))))
            w = smooth(cnt, max(1, int(round(3.0 / grid_step))))
            w = w / (w + n0)
            w[:idxg[cov][0]] = 0
            w[idxg[cov][-1] + 1:] = 0
            ref = w * pseudo + (1 - w) * ref
    # guard: typewell agreement with the horizontal tool in the known zone
    corr = 0.0
    m = np.isfinite(g_at_known)
    if m.sum() > 30 and np.std(g_at_known[m]) > 1e-9 and np.std(gr_k[m]) > 1e-9:
        corr = float(np.corrcoef(A * g_at_known[m] + B, gr_k[m])[0, 1])
        if not np.isfinite(corr):
            corr = 0.0
    return grid, ref, corr

# --------------------------------------------------------- 5. emission core

def _rolling_mean_axis0(X, w):
    """Centered rolling mean along axis 0 via cumsum (edge-shrunk windows)."""
    n = X.shape[0]
    c = np.cumsum(X, axis=0, dtype=np.float64)
    c = np.concatenate([np.zeros((1,) + X.shape[1:]), c], axis=0)
    h = w // 2
    lo = np.clip(np.arange(n) - h, 0, n)
    hi = np.clip(np.arange(n) + h + 1, 0, n)
    return (c[hi] - c[lo]) / (hi - lo).reshape(-1, *([1] * (X.ndim - 1)))

def build_core(arr, t, band=650.0, grid_step=0.5, gr_smooth_ft=3.0,
               type_smooth_ft=1.0, pseudo_smooth_ft=1.5, n0=3.0,
               u_window=None, _ref_cache=None, dc_window_ft=0.0, dc_mode='mean',
               ncc_window_ft=0.0, ncc_shear_max=0.12, ncc_n_shear=7,
               spatial_center=False):
    """Solver-independent per-well quantities, computed once.

    Returns dict with:
      grid       u-state grid (anchor_u +- band)
      R          float32 [n_blind, P] raw |GR - ref| (0 where GR missing)
      prior_dev  float32 [n_blind, P] |u - constant-TVT path| in ft
      k0, z, idx, u_anchor, corr, dmd
    """
    if _ref_cache is not None:
        ref_grid, ref, corr = _ref_cache
    else:
        ref_grid, ref, corr = build_reference(arr, t, grid_step, type_smooth_ft,
                                              pseudo_smooth_ft, n0)
    k0, z, dmd = arr['k0'], arr['z'], arr['dmd']
    u_anchor = arr['anchor_tvt'] + z[k0 - 1]

    ok = np.isfinite(arr['gr'])
    if ok.any():
        gr_s = smooth(np.where(ok, arr['gr'], np.nanmedian(arr['gr'])),
                      max(1, int(round(gr_smooth_ft / dmd))))
        gr_s[~ok] = np.nan
    else:
        gr_s = np.full(arr['n'], np.nan)

    lo, hi = u_anchor - band, u_anchor + band
    if u_window is not None:
        lo = max(lo, u_window[0]); hi = min(hi, u_window[1])
        lo = min(lo, u_anchor - 10); hi = max(hi, u_anchor + 10)  # keep anchor interior
    lo = u_anchor - np.ceil((u_anchor - lo) / grid_step) * grid_step  # anchor on-grid
    grid = np.arange(lo, hi + grid_step, grid_step)
    idx = np.arange(k0, arr['n'])
    tvt_cand = np.clip(grid[None, :] - z[idx][:, None], ref_grid[0], ref_grid[-1])
    g_at = np.interp(tvt_cand, ref_grid, ref)
    obs = gr_s[idx]
    Nc = None
    if ncc_window_ft > 0:
        # Sheared windowed correlation: for shear s (relative dip, ft/ft), the
        # state's reference trace sweeps through the ref profile. Computed per
        # shear via rolling sums on a globally sheared reference matrix, then
        # gathered back with a row-dependent column offset. Nc = min over shears
        # of (1 - corr): best shape match at any plausible local dip.
        w = max(5, int(round(ncc_window_ft / dmd)))
        ok_o = np.isfinite(obs)
        fill = np.nanmedian(obs) if ok_o.any() else 0.0
        x = np.where(ok_o, obs, fill)[:, None]
        mx = _rolling_mean_axis0(x, w)
        vx = np.maximum(_rolling_mean_axis0(x * x, w) - mx ** 2, 0.0)
        sx = np.sqrt(vx)
        s_ref = max(float(np.nanstd(x - mx)), 1e-3)
        md_rel = (np.arange(len(idx)) * dmd)
        P = len(grid)
        cols_base = np.arange(P)
        Nc = None
        for s in np.linspace(-ncc_shear_max, ncc_shear_max, ncc_n_shear):
            tvt_sh = np.clip((grid[None, :] + s * md_rel[:, None]) - z[idx][:, None],
                             ref_grid[0], ref_grid[-1])
            g_s = np.interp(tvt_sh, ref_grid, ref)
            my = _rolling_mean_axis0(g_s, w)
            mxy = _rolling_mean_axis0(x * g_s, w)
            vy = np.maximum(_rolling_mean_axis0(g_s * g_s, w) - my ** 2, 0.0)
            sy = np.sqrt(vy)
            denom = np.maximum(sx, 0.15 * s_ref) * np.maximum(sy, 0.15 * s_ref)
            rho_c = np.clip((mxy - mx * my) / denom, -1.0, 1.0)
            # state u at station i lives at sheared column p - s*md_rel[i]/step
            shift = np.round(s * md_rel / grid_step).astype(np.int64)
            cols = cols_base[None, :] - shift[:, None]
            valid = (cols >= 0) & (cols < P)
            cc = np.take_along_axis(rho_c, np.clip(cols, 0, P - 1), axis=1)
            cc[~valid] = 0.0
            nc_s = (1.0 - cc).astype(np.float32)
            Nc = nc_s if Nc is None else np.minimum(Nc, nc_s)
    if dc_window_ft > 0:
        # drift-cancelling: remove long-window rolling mean (offset drift) and,
        # in 'z' mode, divide by rolling std (gain drift) - both path-independent
        w = max(3, int(round(dc_window_ft / dmd)))
        obs_ok = np.isfinite(obs)
        obs_fill = np.where(obs_ok, obs, np.nanmedian(obs) if obs_ok.any() else 0.0)
        mu_o = _rolling_mean_axis0(obs_fill[:, None], w)[:, 0]
        obs_dc = obs - mu_o
        mu_g = _rolling_mean_axis0(g_at, w)
        g_dc = g_at - mu_g
        if dc_mode == 'z':
            v_o = _rolling_mean_axis0((obs_fill - mu_o)[:, None] ** 2, w)[:, 0]
            v_g = _rolling_mean_axis0(g_dc ** 2, w)
            s_ref = max(float(np.nanstd(obs_dc)), 1e-3)
            sd_o = np.maximum(np.sqrt(np.maximum(v_o, 0)), 0.3 * s_ref)
            sd_g = np.maximum(np.sqrt(np.maximum(v_g, 0)), 0.3 * s_ref)
            R = np.abs(obs_dc[:, None] / sd_o[:, None] - g_dc / sd_g) * s_ref
        else:
            R = np.abs(obs_dc[:, None] - g_dc)
    else:
        R = np.abs(obs[:, None] - g_at)
    R[~np.isfinite(R)] = 0.0                      # missing GR -> uninformative
    center = (arr['anchor_tvt'] + z[idx])
    dip_sp = arr.get('dip_spatial')
    if (spatial_center or CONFIG.get('spatial_prior', False)) and dip_sp is not None:
        md_rel_p = (idx - arr.get('k_last', 0)) * arr['dmd']
        cap = CONFIG.get('spatial_cap', 40.0)
        shift = np.clip(dip_sp * np.maximum(md_rel_p, 0.0), -cap, cap)
        center = center + shift
    prior_dev = np.abs(grid[None, :] - center[:, None])
    if CONFIG.get('field_prior', False) and arr.get('tvt_field') is not None:
        cf_i = arr['field_conf_sta'][idx]
        cen_f = arr['tvt_field'][idx] + z[idx]
        dev_f = np.abs(grid[None, :] - cen_f[:, None])
        ratio = CONFIG.get('field_prior_rho', 0.06) / 0.02
        prior_dev = (prior_dev
                     + (ratio * cf_i)[:, None] * dev_f).astype(np.float32)
    _ked = known_end_dip(arr, CONFIG.get('init_dip_fit_ft', 600.0))
    core_dip, u_fit = (_ked if _ked is not None else (None, None))
    if CONFIG.get('spatial_init', False):
        _dsp = arr.get('dip_spatial')
        if _dsp is not None:
            _c = 0.5 * arr.get('spatial_conf', 1.0)
            core_dip = _dsp if core_dip is None else (1 - _c) * core_dip + _c * _dsp
    _keep = CONFIG.get('w_le', 0) > 0
    return dict(grid=grid, R=R.astype(np.float32), dip_init=core_dip, u_fit=u_fit, Nc=Nc,
                _obs=(obs if _keep else None), _g_at=(g_at if _keep else None),
                prior_dev=prior_dev.astype(np.float32),
                k0=k0, z=z, idx=idx, u_anchor=float(u_anchor),
                corr=corr, dmd=dmd, grid_step=grid_step,
                ref_cache=(ref_grid, ref, corr))


_LE_MU = np.array([16.585415, 16.642612, 0.989071, 14.761854, 1.234139])
_LE_SD = np.array([15.00066, 12.850107, 0.475132, 13.797323, 1.294928])
_LE_W = np.array([-0.397183, -0.324086, -0.075375, -1.052803, -0.182295])
_LE_B = -2.751722

def learned_emission(core):
    """Vectorized learned matchedness emission over (station, state).

    Five features computed via rolling sums, standardized with frozen training
    statistics, combined with frozen logistic weights; emission cost is the
    negative logit scaled to emission units. Requires core built with
    keep_raw pieces (obs, g_at) - computed inline here from R-precursors kept
    in the core when CONFIG['w_le'] > 0.
    """
    obs = core['_obs']; g_at = core['_g_at']
    w = int(CONFIG.get('le_window', 61))
    ok = np.isfinite(obs)
    fill = np.nanmedian(obs[ok]) if ok.any() else 0.0
    x = np.where(ok, obs, fill)[:, None]
    f0 = np.abs(x - g_at)
    f1 = _rolling_mean_axis0(f0, w)
    mu_o = _rolling_mean_axis0(x, w)
    mu_g = _rolling_mean_axis0(g_at, w)
    f3 = np.abs(mu_o - mu_g)
    v_o = np.maximum(_rolling_mean_axis0(x * x, w) - mu_o ** 2, 1e-9)
    v_g = np.maximum(_rolling_mean_axis0(g_at * g_at, w) - mu_g ** 2, 1e-9)
    f4 = np.abs(0.5 * (np.log(v_o) - np.log(v_g)))
    mxy = _rolling_mean_axis0(x * g_at, w)
    corr = (mxy - mu_o * mu_g) / np.sqrt(v_o * v_g)
    f2 = 1.0 - np.clip(corr, -1.0, 1.0)
    logit = _LE_B
    for F, m, s, wt in ((f0, _LE_MU[0], _LE_SD[0], _LE_W[0]),
                        (f1, _LE_MU[1], _LE_SD[1], _LE_W[1]),
                        (f2, _LE_MU[2], _LE_SD[2], _LE_W[2]),
                        (f3, _LE_MU[3], _LE_SD[3], _LE_W[3]),
                        (f4, _LE_MU[4], _LE_SD[4], _LE_W[4])):
        logit = logit + wt * ((F - m) / s)
    E = (-np.float32(CONFIG.get('le_scale', 12.0)) * logit).astype(np.float32)
    E[~ok, :] = 0.0
    return E - E.min(axis=1, keepdims=True)

def derive_emissions(core, emis_clip=40.0, rho=0.02, ncc_scale=0.0,
                     ncc_add_level=False):
    """Cheap per-member emission matrix from the shared core.

    ncc_scale > 0 replaces the level term with the windowed-correlation term
    (shape matching, invariant to slowly varying gain/offset)."""
    if ncc_scale > 0:
        if core.get('Nc') is None:
            raise GuardError('ncc_core_missing')
        E = np.float32(ncc_scale) * core['Nc']
        if ncc_add_level:
            E = E + np.minimum(core['R'], np.float32(emis_clip))
    else:
        E = np.minimum(core['R'], np.float32(emis_clip))
    if rho > 0:
        E = E + np.float32(rho) * core['prior_dev']
    return E

def block_reduce(core, E, block_ft=30.0):
    """Average station emissions into MD blocks. Returns (Eb, nb, block)."""
    block = max(4, int(round(block_ft / core['dmd'])))
    n = E.shape[0]
    nb = n // block
    if nb < 2:
        raise GuardError('too_short_for_blocks')
    Eb = E[:nb * block].reshape(nb, block, E.shape[1]).mean(axis=1) * block
    return Eb, nb, block

# --------------------------------------------------------------- 6. solvers

def _u_path_to_pred(core, us, nb, block):
    """Interpolate block-node u values to stations; TVT = u - Z."""
    xs = core['k0'] + np.arange(nb + 1) * block
    stations = np.arange(core['k0'], core['k0'] + len(core['idx']))
    u_path = np.interp(stations, xs, us)
    pred = np.full(core['k0'] + len(core['idx']), np.nan)
    pred[:core['k0']] = np.nan                      # caller fills known zone
    pred[core['idx']] = u_path - core['z'][core['idx']]
    return pred






_SPATIAL_MAP = None

def build_spatial_map(exclude=()):
    """Structural map from training wells: per-well (x, y, unit heading, blind dip).
    Uses training truth (allowed at inference). Cached at module level."""
    global _SPATIAL_MAP
    ex = set(exclude)
    rows = []
    for f in sorted(glob.glob(os.path.join(DATA, 'train', '*__horizontal_well.csv'))):
        w = os.path.basename(f).split('__')[0]
        if w in ex:
            continue
        try:
            h = pd.read_csv(f, usecols=['X', 'Y', 'Z', 'MD', 'TVT', 'TVT_input'])
        except Exception:
            continue
        blind = h.TVT_input.isna().values
        if blind.sum() < 100 or (~blind).sum() < 50:
            continue
        u = h.TVT.values + h.Z.values
        ub = u[blind]; mb = h.MD.values[blind]
        m = np.isfinite(ub) & np.isfinite(mb)
        if m.sum() < 100:
            continue
        ub, mb = ub[m], mb[m]
        A = np.vstack([mb - mb.mean(), np.ones(len(mb))]).T
        try:
            slope = float(np.linalg.lstsq(A, ub, rcond=None)[0][0])
        except Exception:
            continue
        hx = float(h.X.values[-1] - h.X.values[0])
        hy = float(h.Y.values[-1] - h.Y.values[0])
        nrm = (hx * hx + hy * hy) ** 0.5
        if not (np.isfinite(slope) and nrm > 1e-6):
            continue
        rows.append((float(np.nanmedian(h.X)), float(np.nanmedian(h.Y)),
                     hx / nrm, hy / nrm, float(np.clip(slope, -0.2, 0.2)), w))
    if rows:
        arr = np.array([r[:5] for r in rows], dtype=float)
        _SPATIAL_MAP = dict(xy=arr[:, :2], h=arr[:, 2:4], dip=arr[:, 4],
                            names=[r[5] for r in rows])
    else:
        _SPATIAL_MAP = dict(xy=np.zeros((0, 2)), h=np.zeros((0, 2)),
                            dip=np.zeros(0), names=[])
    return _SPATIAL_MAP


def predict_spatial_dip(h_df, self_name=None):
    """(dip, nn_dist) from the local structural-gradient fit at this well's
    location and heading; (None, nn_dist) when unavailable or out of footprint."""
    M = _SPATIAL_MAP
    if M is None or len(M['dip']) < 5:
        return None, None
    if 'X' not in h_df.columns or 'Y' not in h_df.columns:
        return None, None
    x = float(np.nanmedian(h_df.X)); y = float(np.nanmedian(h_df.Y))
    if not (np.isfinite(x) and np.isfinite(y)):
        return None, None
    keep = np.ones(len(M['dip']), dtype=bool)
    if self_name is not None and self_name in M['names']:
        keep[M['names'].index(self_name)] = False
    xy = M['xy'][keep]; hh_all = M['h'][keep]; dips = M['dip'][keep]
    if len(dips) < 5:
        return None, None
    d2 = (xy[:, 0] - x) ** 2 + (xy[:, 1] - y) ** 2
    nn = float(np.sqrt(d2.min()))
    if nn > CONFIG.get('spatial_max_nn', 30000.0):
        return None, nn
    idx = np.argsort(d2)[:int(CONFIG.get('spatial_k', 25))]
    wgt = 1.0 / (np.sqrt(d2[idx]) + CONFIG.get('spatial_soft', 3000.0))
    hx = float(h_df.X.values[-1] - h_df.X.values[0])
    hy = float(h_df.Y.values[-1] - h_df.Y.values[0])
    nrm = (hx * hx + hy * hy) ** 0.5
    sw = np.sqrt(wgt)
    try:
        if nrm > 1e-6:
            g, *_ = np.linalg.lstsq(hh_all[idx] * sw[:, None], dips[idx] * sw,
                                    rcond=None)
            dip = float(np.array([hx / nrm, hy / nrm]) @ g)
        else:
            dip = float(np.sum(wgt * dips[idx]) / np.sum(wgt))
    except Exception:
        dip = float(np.sum(wgt * dips[idx]) / np.sum(wgt))
    if not np.isfinite(dip):
        return None, nn
    return float(np.clip(dip, -0.2, 0.2)), nn


_UFIELD = None

def build_ufield(exclude=()):
    """Structural point field from training wells, datum-aligned by typewell
    formation tops (primary top with median-spacing fallbacks)."""
    global _UFIELD
    from scipy.spatial import cKDTree
    ex = set(exclude)
    tops_all = {}
    for f in sorted(glob.glob(os.path.join(DATA, 'train', '*__typewell.csv'))):
        w = os.path.basename(f).split('__')[0]
        if w in ex:
            continue
        try:
            t = pd.read_csv(f, usecols=['TVT', 'Geology'])
        except Exception:
            continue
        labs = [str(x) for x in t['Geology'].tolist()]
        tp = {}; prev = None
        for tvt, g in zip(t['TVT'].values, labs):
            if g not in ('nan', 'None', '') and g != prev and g not in tp:
                tp[g] = float(tvt)
            if g not in ('nan', 'None', ''):
                prev = g
        if tp:
            tops_all[w] = tp
    if not tops_all:
        _UFIELD = None
        return None
    from collections import Counter
    cnt = Counter(g for tp in tops_all.values() for g in tp)
    primary = cnt.most_common(1)[0][0]
    spac = {}
    for g in cnt:
        if g == primary:
            continue
        ds = [tp[primary] - tp[g] for tp in tops_all.values()
              if primary in tp and g in tp]
        if len(ds) >= 20:
            spac[g] = float(np.median(ds))
    offs = {}
    for w, tp in tops_all.items():
        if primary in tp:
            offs[w] = tp[primary]
        else:
            for g, s in sorted(spac.items(), key=lambda kv: -cnt[kv[0]]):
                if g in tp:
                    offs[w] = tp[g] + s
                    break
    pts = []; us = []
    for f in sorted(glob.glob(os.path.join(DATA, 'train',
                                           '*__horizontal_well.csv'))):
        w = os.path.basename(f).split('__')[0]
        if w in ex or w not in offs:
            continue
        try:
            h = pd.read_csv(f, usecols=['X', 'Y', 'Z', 'TVT'])
        except Exception:
            continue
        m = (np.isfinite(h.X.values) & np.isfinite(h.Y.values)
             & np.isfinite(h.Z.values) & np.isfinite(h.TVT.values))
        if m.sum() < 200:
            continue
        pts.append(np.column_stack([h.X.values[m][::8], h.Y.values[m][::8]]))
        us.append((h.TVT.values[m] + h.Z.values[m])[::8] - offs[w])
    if not pts:
        _UFIELD = None
        return None
    P = np.vstack(pts); Uv = np.concatenate(us)
    n_train = len(Uv)
    wid = [''] * n_train
    if CONFIG.get('field_aug', False):
        _tree_tr = cKDTree(P)

        def _est_tr(xq, yq, k=40, soft=400.0):
            dd, idx = _tree_tr.query(np.column_stack([xq, yq]), k=k)
            wq = 1.0 / (dd + soft) ** 2
            sq = wq.sum(1)
            e = np.einsum('nk,nk->n', wq, Uv[:n_train][idx]) / np.maximum(sq, 1e-12)
            e[sq <= 1e-12] = np.nan
            return e
        for f in sorted(glob.glob(os.path.join(DATA, 'test',
                                               '*__horizontal_well.csv'))):
            w = os.path.basename(f).split('__')[0]
            try:
                hh = pd.read_csv(f, usecols=['X', 'Y', 'Z', 'TVT_input'])
            except Exception:
                continue
            m = (hh.TVT_input.notna().values & np.isfinite(hh.X.values)
                 & np.isfinite(hh.Y.values) & np.isfinite(hh.Z.values))
            if m.sum() < 100:
                continue
            uk = (hh.TVT_input.values + hh.Z.values)[m]
            try:
                ek = _est_tr(hh.X.values[m], hh.Y.values[m])
            except Exception:
                continue
            okk = np.isfinite(ek)
            if okk.sum() < 50:
                continue
            d_w = float(np.median(uk[okk] - ek[okk]))
            P = np.vstack([P, np.column_stack([hh.X.values[m][::4],
                                               hh.Y.values[m][::4]])])
            Uv = np.concatenate([Uv, uk[::4] - d_w])
            wid.extend([w] * len(uk[::4]))
    _UFIELD = dict(tree=cKDTree(P), U=Uv, n=len(Uv), n_train=n_train,
                   wid=np.array(wid),
                   istrain=np.arange(len(Uv)) < n_train)
    return _UFIELD


def _field_query(xq, yq, self_well=None):
    F = _UFIELD
    k = int(CONFIG.get('field_k', 40)); soft = CONFIG.get('field_soft', 400.0)
    dd, idx = F['tree'].query(np.column_stack([xq, yq]), k=k)
    istr = F.get('istrain')
    if istr is None:
        wgt = 1.0 / (dd + soft) ** 2
        d_conf = dd[:, 0]
    else:
        mask_self = ((F['wid'][idx] == self_well)
                     if self_well is not None else np.zeros(idx.shape, bool))
        d_train = np.where(mask_self | ~istr[idx], np.inf, dd).min(1)
        Lg = CONFIG.get('field_aug_Lg', 1200.0)
        gate = 1.0 - np.exp(-(np.minimum(d_train, 1e5) / Lg) ** 2)
        bw = np.where(istr[idx], 1.0,
                      CONFIG.get('field_aug_w', 0.35) * gate[:, None])
        wgt = np.where(mask_self, 0.0, bw / (dd + soft) ** 2)
        d_conf = d_train
    s = wgt.sum(1)
    est = np.einsum('nk,nk->n', wgt, F['U'][idx]) / np.maximum(s, 1e-12)
    var = np.einsum('nk,nk->n', wgt,
                    (F['U'][idx] - est[:, None]) ** 2) / np.maximum(s, 1e-12)
    est[s <= 1e-12] = np.nan
    return est, d_conf, np.sqrt(np.maximum(var, 0))


def field_blend(h, pred, diag, arr=None):
    """Per-station confidence blend of the structural-field prediction."""
    if _UFIELD is None or not CONFIG.get('field_blend', False):
        return pred
    if 'X' not in h.columns or 'Y' not in h.columns:
        return pred
    b = h.TVT_input.isna().values
    m_all = (np.isfinite(h.X.values) & np.isfinite(h.Y.values)
             & np.isfinite(h.Z.values))
    ku = (~b) & m_all & np.isfinite(h.TVT_input.values)
    bu = b & m_all
    if ku.sum() < 100 or bu.sum() < 200:
        return pred
    _sw = h.attrs.get('well') if hasattr(h, 'attrs') else None
    est_k, dk, sk = _field_query(h.X.values[ku], h.Y.values[ku], self_well=_sw)
    u_k = h.TVT_input.values[ku] + h.Z.values[ku]
    okk = np.isfinite(est_k)
    if okk.sum() < 50:
        return pred
    resid = u_k[okk] - est_k[okk]
    off = float(np.median(resid))
    mad = float(np.median(np.abs(resid - off)))
    est_b, db, sb = _field_query(h.X.values[bu], h.Y.values[bu], self_well=_sw)
    tvt_f = est_b + off - h.Z.values[bu]
    conf = (CONFIG.get('field_wmax', 0.55)
            * np.exp(-db / CONFIG.get('field_Ld', 800.0))
            * np.exp(-sb / CONFIG.get('field_Ls', 8.0))
            * np.exp(-mad / CONFIG.get('field_Lm', 6.0)))
    if CONFIG.get('field_ivar', False) and arr is not None \
            and arr.get('_branch_paths') and len(arr['_branch_paths']) >= 3:
        paths = np.stack(arr['_branch_paths'])
        spread = np.median(np.abs(paths - np.median(paths, axis=0)), axis=0)[bu]
        sig_t = np.maximum(1.0, CONFIG.get('field_sig_trk', 1.5) * spread)
        sig_f = np.maximum(2.0, (CONFIG.get('field_sig_fld', 0.5) * sb
                                 + db / CONFIG.get('field_sig_d', 400.0) + mad))
        w_iv = sig_t ** 2 / (sig_t ** 2 + sig_f ** 2)
        qual = (np.exp(-db / CONFIG.get('field_Ld', 800.0))
                * np.exp(-mad / CONFIG.get('field_Lm', 6.0)))
        conf = np.minimum(0.95, w_iv * qual)
        if CONFIG.get('field_floor_w', 0.0) > 0:
            floor = (CONFIG['field_floor_w']
                     * np.exp(-(db / CONFIG.get('field_floor_Ln', 1200.0)) ** 2)
                     * np.exp(-mad / CONFIG.get('field_Lm', 12.0)))
            dis = np.abs(pred[bu] - tvt_f)
            dis = np.where(np.isfinite(dis), dis, 0.0)
            floor = floor / (1.0 + np.exp(-(dis - CONFIG.get('field_floor_D0', 9.0))
                                          / CONFIG.get('field_floor_Ds', 3.0)))
            conf = np.maximum(conf, np.minimum(floor, 0.9))
        diag['ivar_wmean'] = round(float(np.nanmean(conf)), 3)
    _Lt = CONFIG.get('field_Lt', 0.0)
    if _Lt and _Lt > 0:
        _k_last = int(np.where(~b)[0][-1]) if (~b).any() else 0
        _mdrel = np.where(bu)[0].astype(float) - _k_last
        conf = conf * (1.0 - np.exp(-np.maximum(_mdrel, 0.0) / _Lt))
    okb = np.isfinite(tvt_f)
    if not okb.any():
        return pred
    pf = pred.copy()
    idx_b = np.where(bu)[0][okb]
    cf = np.clip(conf[okb], 0.0, CONFIG.get('field_wmax', 0.55))
    pf[idx_b] = (1 - cf) * pf[idx_b] + cf * tvt_f[okb]
    diag['field_mad'] = round(mad, 2)
    diag['field_conf'] = round(float(cf.mean()), 3)
    return pf

def _effective_anchor(core):
    """Raw anchor, optionally replaced by the fitted known-zone boundary value
    (clamped to anchor_fit_clamp ft of the raw anchor)."""
    ua = core['u_anchor']
    if CONFIG.get('anchor_fit', False):
        uf = core.get('u_fit')
        if uf is not None and np.isfinite(uf):
            c = CONFIG.get('anchor_fit_clamp', 10.0)
            ua = float(np.clip(uf, ua - c, ua + c))
    return ua

def _maybe_graze_redo(p, arr2, t, ref_b, dc_w, tube):
    """If a branch path grazes the shared scout tube it was solved in, redo that
    branch on the full band. Gated by CONFIG['branch_graze_redo']."""
    if not CONFIG.get('branch_graze_redo', False):
        return p
    try:
        blind = arr2['blind']
        u_b = p[blind] + arr2['z'][blind]
        g = CONFIG['tube_graze']
        if np.nanmin(u_b) > tube[0] + g and np.nanmax(u_b) < tube[1] - g:
            return p
        core_b = build_core(arr2, t, _ref_cache=ref_b, dc_window_ft=dc_w,
                            dc_mode=CONFIG['dc_mode'])
        E = derive_emissions(core_b, emis_clip=40.0, rho=0.02)
        Eb, nb, block = block_reduce(core_b, E)
        us = solve_viterbi(core_b, Eb, nb, block, **SOLVE)
        return _u_path_to_pred(core_b, us, nb, block)
    except Exception:
        return p

def known_end_dip(arr, fit_ft=600.0, min_ft=150.0):
    """Robust structural dip (d u / d md, ft/ft) at the end of the known zone.
    Returns None when the known zone is too short or the fit is degenerate."""
    kidx = np.where(~arr['blind'])[0]
    if len(kidx) < 10:
        return None
    n_fit = int(round(fit_ft / arr['dmd']))
    kidx = kidx[-max(int(round(min_ft / arr['dmd'])), min(n_fit, len(kidx))):]
    tvt_k = arr['tvt_in'][kidx]
    m = np.isfinite(tvt_k)
    if m.sum() < 10:
        return None
    x = kidx[m] * arr['dmd']
    u = tvt_k[m] + arr['z'][kidx][m]
    A = np.vstack([x - x.mean(), np.ones(m.sum())]).T
    try:
        sol, res, *_ = np.linalg.lstsq(A, u, rcond=None)
    except Exception:
        return None
    slope = float(sol[0])
    if not np.isfinite(slope):
        return None
    x_bnd = (kidx[-1] + 1) * arr['dmd']
    u_bnd = float(sol[0] * (x_bnd - x.mean()) + sol[1])
    return float(np.clip(slope, -0.2, 0.2)), (u_bnd if np.isfinite(u_bnd) else None)

def _transition_maps(P, D):
    """Precompute gather maps for banded (u, dip) transitions.

    Forward semantics: state (p, j) receives from (p - D[j], j - dd), dd in {-1,0,1}.
    Backward semantics: (p, jj) receives from (p + D[j], j) with j = jj + dd.
    Returns dict of per-dd (rows, cols, valid) index arrays of shape [P, nd].
    """
    nd = len(D)
    ar_p = np.arange(P)[:, None]
    ar_j = np.arange(nd)
    fwd, bwd = {}, {}
    for dd in (-1, 0, 1):
        cols_f = ar_j - dd
        okc_f = (cols_f >= 0) & (cols_f < nd)
        rows_f = ar_p - D[None, :]
        okr_f = (rows_f >= 0) & (rows_f < P)
        fwd[dd] = (np.clip(rows_f, 0, P - 1), np.clip(cols_f, 0, nd - 1)[None, :],
                   okr_f & okc_f[None, :])
        cols_b = ar_j + dd
        okc_b = (cols_b >= 0) & (cols_b < nd)
        cols_bc = np.clip(cols_b, 0, nd - 1)
        rows_b = ar_p + D[None, cols_bc]
        okr_b = (rows_b >= 0) & (rows_b < P)
        bwd[dd] = (np.clip(rows_b, 0, P - 1), cols_bc[None, :], okr_b & okc_b[None, :])
    return fwd, bwd

def _sliding_min(a, half):
    """Per-column sliding minimum over a window of +-half along axis 0."""
    try:
        from scipy.ndimage import minimum_filter1d
        return minimum_filter1d(a, size=2 * half + 1, axis=0, mode='nearest')
    except Exception:
        out = a.copy()
        for s in range(1, half + 1):
            out[s:] = np.minimum(out[s:], a[:-s])
            out[:-s] = np.minimum(out[:-s], a[s:])
        return out

def solve_viterbi(core, Eb, nb, block, kappa=300.0, dip_max_steps=14,
                  jump_cost=0.0, jump_max_ft=100.0):
    grid = core['grid']; P = len(grid); step = core['grid_step']
    D = np.arange(-dip_max_steps, dip_max_steps + 1)
    nd = len(D)
    fwd, _ = _transition_maps(P, D)
    INF = 1e18
    J = int(round(jump_max_ft / step)) if jump_cost > 0 else 0
    cost = np.full((P, nd), INF)
    ua = _effective_anchor(core)
    s0 = int(round((ua - grid[0]) / step))
    cost[s0, :] = 0.0
    di = core.get('dip_init')
    if di is not None and CONFIG.get('init_dip_pen', 0) > 0:
        j_star = np.clip(round(di * block * core['dmd'] / step), D[0], D[-1])
        cost[s0, :] = CONFIG['init_dip_pen'] * np.abs(D - j_star)
    ptr = np.zeros((nb, P, nd), dtype=np.int8)
    jflag = np.zeros((nb, P, nd), dtype=bool) if J else None
    for ib in range(nb):
        best = None; best_dd = None
        for c, dd in enumerate((-1, 0, 1)):
            rows, cols, ok = fwd[dd]
            G = cost[rows, cols] + kappa * abs(dd)
            G[~ok] = INF
            if best is None:
                best, best_dd = G, np.zeros((P, nd), dtype=np.int8)
            else:
                take = G < best
                best = np.where(take, G, best)
                best_dd = np.where(take, np.int8(c), best_dd)
        if J:
            # fault option: arrive at (p, j) from (q, j), |q-p|<=J, fixed cost
            Gj = _sliding_min(cost, J) + jump_cost
            take = Gj < best
            best = np.where(take, Gj, best)
            jflag[ib] = take
        cost = best + Eb[ib].astype(np.float64)[:, None]
        ptr[ib] = best_dd
    p, j = np.unravel_index(int(np.argmin(cost)), (P, nd))
    us = np.zeros(nb + 1)
    us[nb] = grid[p]
    # rebuild forward costs for jump-source recovery is avoided by local search:
    # during backtrack, a jump block picks the best source within the window.
    # We re-run forward storing per-block pre-emission costs for exact recovery.
    if J:
        # second pass to store costs per block (memory nb*P*nd float32)
        costs_hist = np.zeros((nb, P, nd), dtype=np.float32)
        cost2 = np.full((P, nd), INF); cost2[s0, :] = 0.0
        for ib in range(nb):
            best = None
            for c, dd in enumerate((-1, 0, 1)):
                rows, cols, ok = fwd[dd]
                G = cost2[rows, cols] + kappa * abs(dd)
                G[~ok] = INF
                best = G if best is None else np.minimum(best, G)
            Gj = _sliding_min(cost2, J) + jump_cost
            best = np.minimum(best, Gj)
            costs_hist[ib] = cost2.astype(np.float32)
            cost2 = best + Eb[ib].astype(np.float64)[:, None]
    for ib in range(nb - 1, -1, -1):
        if J and jflag[ib, p, j]:
            lo, hi = max(0, p - J), min(P, p + J + 1)
            p = int(lo + np.argmin(costs_hist[ib, lo:hi, j]))
            us[ib] = grid[p]
            continue
        dd = int(ptr[ib, p, j]) - 1
        p = int(np.clip(p - D[j], 0, P - 1))
        j = int(np.clip(j - dd, 0, nd - 1))
        us[ib] = grid[p]
    return us

def solve_posterior(core, Eb, nb, block, kappa=300.0, dip_max_steps=14, temp=8.0,
                    decode='mean'):
    grid = core['grid']; P = len(grid); step = core['grid_step']
    D = np.arange(-dip_max_steps, dip_max_steps + 1)
    nd = len(D)
    fwd, bwd = _transition_maps(P, D)
    NEG = -1e18
    Ebt = Eb.astype(np.float64) / temp
    kap = kappa / temp

    def prop(lp, maps):
        out = None
        for dd in (-1, 0, 1):
            rows, cols, ok = maps[dd]
            G = lp[rows, cols] - kap * abs(dd)
            G[~ok] = NEG
            out = G if out is None else np.logaddexp(out, G)
        return out

    alpha = np.full((nb + 1, P, nd), NEG, dtype=np.float64)
    ua = _effective_anchor(core)
    s0 = int(round((ua - grid[0]) / step))
    alpha[0, s0, :] = 0.0
    di = core.get('dip_init')
    if di is not None and CONFIG.get('init_dip_pen', 0) > 0:
        j_star = np.clip(round(di * block * core['dmd'] / step), D[0], D[-1])
        alpha[0, s0, :] = -(CONFIG['init_dip_pen'] / temp) * np.abs(D - j_star)
    for ib in range(nb):
        a = prop(alpha[ib], fwd) - Ebt[ib][:, None]
        alpha[ib + 1] = a - a.max()
    beta = np.full((nb + 1, P, nd), NEG, dtype=np.float64)
    beta[nb] = 0.0
    for ib in range(nb - 1, -1, -1):
        b = prop(beta[ib + 1] - Ebt[ib][:, None], bwd)
        beta[ib] = b - b.max()

    us = np.zeros(nb + 1)
    for ib in range(nb + 1):
        lp = alpha[ib] + beta[ib]
        lp -= lp.max()
        pr = np.exp(lp).sum(axis=1)
        pr /= pr.sum()
        if decode == 'median':
            us[ib] = float(grid[np.searchsorted(np.cumsum(pr), 0.5)])
        else:
            us[ib] = float(pr @ grid)
    us[0] = core['u_anchor']
    return us

# ----------------------------------------------------------- 7. orchestrator

CONFIG = dict(
    members_a=[('vit', dict(emis_clip=40.0, rho=0.02)),
               ('vit', dict(emis_clip=25.0, rho=0.02)),
               ('post', dict(emis_clip=40.0, rho=0.02))],
    member_b=('vit', dict(emis_clip=40.0, rho=0.0)),
    w_b=0.4,                 # prior-free member weight (0.5 measured -1 ft on hidden LB)
    solve=dict(kappa=300.0, dip_max_steps=14),
    scout=dict(grid_step=1.0, emis_clip=40.0, rho=0.02),
    tube_margin=150.0,       # around scout path; boundary-graze triggers full-band redo
    tube_graze=5.0,
    em_weight=0.3,           # damping for pass-1 pairs added to the pseudo-typewell
    em_div_guard=40.0,       # ft; keep pass 1 if refinement diverges beyond this
    em_min_pairs=100,
    init_dip_pen=300.0,      # per-step penalty anchoring initial dip to known-zone trend
    anchor_fit=False,        # falsified: raw handoff anchor is better
    em_iters=1,              # EM refinement iterations (2 = decayed second pass)
    spatial_prior=False,     # falsified as shared prior (error compounds with MD)
    spatial_init=True,       # blend spatial dip into initial-dip anchoring
    spatial_k=25,            # neighbors for the local gradient fit
    spatial_soft=3000.0,     # ft distance softening for neighbor weights
    spatial_max_nn=60000.0,  # ft hard gate; the confidence taper handles mid-range
    w_spatial=0.15,          # spatial-path branch weight (sequential, after dctw)
    spatial_rho=0.02,        # prior strength for the spatial branch (std member level)
    spatial_cap=40.0,        # ft cap on the sloped-center shift
    spatial_conf_L=15000.0,  # ft e-folding of spatial confidence (calibrated on cluster holdout)
    formation_affine=False,  # per-Geology-label typewell calibration
    formation_min_pairs=40,  # known-zone pairs needed to fit a formation's affine
    formation_shrink=60.0,   # count-shrinkage toward the global affine
    formation_fade_ft=12.0,  # crossfade of (a, b) across formation boundaries
    w_le=0.0,                # learned-emission branch weight (sequential)
    le_window=61,            # samples in the matchedness window (matches training)
    le_scale=6.0,            # logit -> emission-cost scale (hybrid regime)
    field_blend=True,        # structural-field per-station blend (post-branches)
    field_k=40,              # neighbors per field query
    field_soft=400.0,        # ft IDW softening
    field_wmax=0.85,         # max per-station blend weight (swept; interior optimum)
    field_Ld=1500.0,         # ft e-folding: distance to nearest field sample
    field_Ls=15.0,           # ft e-folding: local field dispersion
    field_Lm=12.0,           # ft e-folding: known-zone field-fit MAD
    field_Lt=800.0,          # ft ramp-in of blend weight past the anchor (heel protection)
    field_ivar=True,         # inverse-variance fusion using branch disagreement
    field_sig_trk=2.5,       # tracker sigma per ft of branch spread
    field_sig_fld=0.5,       # field sigma per ft of local dispersion
    field_sig_d=400.0,       # ft of nn distance per +1 ft field sigma
    field_aug=False,         # LB-FALSIFIED v20: datum calibration unreliable at distance
    field_aug_w=0.35,        # max augmented-point weight
    field_aug_Lg=1200.0,     # ft: training-support distance where augmentation ramps in
    field_floor_w=0.0,       # consensus-override floor on field weight (0 = off)
    field_floor_Ln=1200.0,   # ft: floor's support-distance scale
    field_floor_D0=9.0,      # ft: tracker-field disagreement where floor engages
    field_floor_Ds=3.0,      # ft: engagement softness
    ref_grid_step=0.5,       # LB-FALSIFIED at 0.25 (v21: gap +0.22); 0.5 canonical
    field_prior=False,       # FALSIFIED ON LB (10.20 vs 9.38): discrete rung-flips; field use must stay proportional
    field_prior_rho=0.06,    # extra prior strength at confidence 1

    anchor_fit_clamp=10.0,   # max ft the fitted anchor may move from the raw anchor
    init_dip_fit_ft=600.0,   # trailing known-zone length for the dip fit
    branch_graze_redo=True,  # redo aux branch full-band if its path grazes the tube
    dev_clip=250.0,          # ft around constant path; catastrophe insurance only
    min_corr=0.3,            # known-zone GR/typewell agreement guard
    dc_window_ft=1200.0,     # drift-cancelling member: rolling-mean window along MD
    dc_member=('vit', dict(emis_clip=40.0, rho=0.02)),
    w_dc=0.40,               # drift-cancel branch weight (v22: corrupted-gate retune)
    recal=False,             # path-dependent recal: falsified (circular); keep off
    dc_mode='mean',          # 'mean' cancels offset drift; 'z' also cancels gain drift
    w_tw=0.28,               # typewell-only branch weight (v22: corrupted-gate retune)
    w_dctw=0.10,             # drift-cancelling on typewell-only reference
    recal_window_ft=1200.0,
    recal_damp=0.6,
)
# Back-compat aliases (kept so experiment scripts keep running)
MEMBERS_A = CONFIG['members_a']; MEMBER_B = CONFIG['member_b']
W_B = CONFIG['w_b']; SOLVE = CONFIG['solve']
EM_WEIGHT = CONFIG['em_weight']; EM_DIV_GUARD = CONFIG['em_div_guard']

def rolling_affine_correction(arr, ref, p1, window_ft=1200.0, damp=0.6,
                              a_lim=(0.6, 1.6), b_lim=30.0, min_pts=80):
    """Estimate slowly-varying gain/offset drift of blind-zone GR relative to the
    reference evaluated along the pass-1 path; return corrected copies of
    (gr, gr_raw). Fits gr ~ a*ref + b in overlapping windows (robust trimmed LS),
    damps toward identity, clamps, and interpolates between window centers.
    Known-zone samples are never modified."""
    ref_grid, ref_g, _ = ref
    blind_idx = np.where(arr['blind'])[0]
    if len(blind_idx) < 3 * min_pts:
        return arr['gr'], arr['gr_raw']
    g_path = np.interp(np.clip(p1[blind_idx], ref_grid[0], ref_grid[-1]),
                       ref_grid, ref_g)
    gr_b = arr['gr'][blind_idx]
    w = max(3, int(round(window_ft / arr['dmd'])))
    step = max(1, w // 2)
    centers, a_s, b_s = [], [], []
    for s in range(0, len(blind_idx) - w + 1, step):
        sl = slice(s, s + w)
        x, y = g_path[sl], gr_b[sl]
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < min_pts or np.std(x[m]) < 1e-6:
            a, b = 1.0, 0.0
        else:
            a, b = _fit_affine(x, y, min_pts=min_pts)
            a = 1.0 + damp * (np.clip(a, *a_lim) - 1.0)
            b = damp * np.clip(b, -b_lim, b_lim)
        centers.append(s + w / 2); a_s.append(a); b_s.append(b)
    if not centers:
        return arr['gr'], arr['gr_raw']
    pos = np.arange(len(blind_idx), dtype=float)
    a_i = np.interp(pos, centers, a_s)
    b_i = np.interp(pos, centers, b_s)
    a_i = np.maximum(a_i, 1e-3)
    gr = arr['gr'].copy(); gr_raw = arr['gr_raw'].copy()
    gr[blind_idx] = (gr[blind_idx] - b_i) / a_i
    gr_raw[blind_idx] = (gr_raw[blind_idx] - b_i) / a_i
    return gr, gr_raw

def detect_z_sign(arr, win_ft=301.0):
    known = np.isfinite(arr['tvt_in'])
    if known.sum() < 400:
        return 1.0
    tv = arr['tvt_in'][known]
    z = arr['z'][known]
    w = max(3, int(round(win_ft / arr['dmd'])))
    thf = tv - smooth(tv, w)
    zhf = z - smooth(z, w)
    if thf.std() * zhf.std() < 1e-12:
        return 1.0
    c = float(np.corrcoef(thf, zhf)[0, 1])
    return 1.0 if (not np.isfinite(c) or c < 0) else -1.0

def _constant_fill(h):
    """Constant-TVT prediction that NEVER returns NaN in the blind zone."""
    tvt_in = h['TVT_input'].to_numpy(dtype=float)
    pred = tvt_in.copy()
    blind = ~np.isfinite(tvt_in)
    known = tvt_in[np.isfinite(tvt_in)]
    if len(known):
        k0 = int(np.argmax(blind))
        fill = known[k0 - 1] if (k0 > 0 and np.isfinite(tvt_in[k0 - 1])) else known[-1]
    else:
        fill = 0.0                                   # replaced by typewell median below
    pred[blind] = fill
    return pred

def predict_constant(h, t=None):
    pred = _constant_fill(h)
    if not np.isfinite(pred).all() or (t is not None and
                                       not np.isfinite(h['TVT_input']).any()):
        # last resort: middle of the typewell's TVT range
        fill = float(t['TVT'].median()) if t is not None else 0.0
        pred[~np.isfinite(pred)] = fill
        if not np.isfinite(h['TVT_input']).any():
            pred[:] = fill
    return pred

def _ensemble_pred(arr, t, ref, tube=None):
    """Scout -> tube -> 4-member ensemble; full-band redo if tube grazed.
    Pass a precomputed tube to skip the scout (used by the EM second pass)."""
    if tube is None:
        sc = CONFIG['scout']
        scout = build_core(arr, t, grid_step=sc['grid_step'], _ref_cache=ref)
        E = derive_emissions(scout, emis_clip=sc['emis_clip'], rho=sc['rho'])
        Eb, nb, block = block_reduce(scout, E)
        us0 = solve_viterbi(scout, Eb, nb, block, kappa=SOLVE['kappa'],
                            dip_max_steps=max(2, SOLVE['dip_max_steps'] // 2))
        m = CONFIG['tube_margin']
        tube = (float(us0.min()) - m, float(us0.max()) + m)
    core = build_core(arr, t, u_window=tube, _ref_cache=ref)
    wts = np.array([(1 - W_B) / len(MEMBERS_A)] * len(MEMBERS_A) + [W_B])
    def run(c):
        out = []
        for mem in MEMBERS_A + [MEMBER_B]:
            kind, ekw = mem[0], mem[1]
            skw = dict(SOLVE); skw.update(mem[2] if len(mem) > 2 else {})
            E = derive_emissions(c, **ekw)
            Eb, nb, block = block_reduce(c, E)
            us = (solve_viterbi if kind == 'vit' else solve_posterior)(
                c, Eb, nb, block, **skw)
            out.append(_u_path_to_pred(c, us, nb, block))
        return np.stack(out)
    preds = run(core)
    u_paths = preds[:, arr['blind']] + arr['z'][arr['blind']][None, :]
    g = CONFIG['tube_graze']
    if (u_paths.min() < core['grid'][0] + g) or (u_paths.max() > core['grid'][-1] - g):
        preds = run(build_core(arr, t, _ref_cache=ref))
    return np.einsum('m,mn->n', wts, preds), tube

def predict_well(h, t, dev_clip=None, min_corr=None):
    pred, status, _ = predict_well_diag(h, t, dev_clip, min_corr)
    return pred, status

def predict_well_diag(h, t, dev_clip=None, min_corr=None):
    dev_clip = CONFIG['dev_clip'] if dev_clip is None else dev_clip
    min_corr = CONFIG['min_corr'] if min_corr is None else min_corr
    """Guarded ensemble; returns (pred, status, diag). pred finite on blind rows."""
    diag = {}
    const = predict_constant(h, t)
    try:
        arr = prepare_arrays(h)
    except GuardError as g:
        return const, f'fallback_{g.status}', diag
    arr['k_last'] = int(np.where(~arr['blind'])[0][-1]) if (~arr['blind']).any() else 0
    arr['tvt_field'] = None
    if CONFIG.get('field_prior', False) and _UFIELD is not None \
            and 'X' in h.columns and 'Y' in h.columns:
        try:
            _m = (np.isfinite(h.X.values) & np.isfinite(h.Y.values)
                  & np.isfinite(h.Z.values))
            _ku = (~arr['blind']) & _m & np.isfinite(h.TVT_input.values)
            if _ku.sum() >= 100:
                _ek, _dk, _sk = _field_query(h.X.values[_ku], h.Y.values[_ku])
                _uk = (h.TVT_input.values + h.Z.values)[_ku]
                _ok = np.isfinite(_ek)
                if _ok.sum() >= 50:
                    _r = _uk[_ok] - _ek[_ok]
                    _off = float(np.median(_r))
                    _mad = float(np.median(np.abs(_r - _off)))
                    _ea, _da, _sa = _field_query(h.X.values, h.Y.values)
                    _tvtf = _ea + _off - h.Z.values
                    _cf = (np.exp(-_da / CONFIG.get('field_Ld', 1500.0))
                           * np.exp(-_sa / CONFIG.get('field_Ls', 15.0))
                           * np.exp(-_mad / CONFIG.get('field_Lm', 12.0)))
                    _cf = np.where(np.isfinite(_tvtf) & _m, _cf, 0.0)
                    arr['tvt_field'] = np.where(np.isfinite(_tvtf), _tvtf, 0.0)
                    arr['field_conf_sta'] = _cf.astype(np.float64)
        except Exception:
            arr['tvt_field'] = None
    if (CONFIG.get('spatial_prior', False) or CONFIG.get('spatial_init', False)
            or CONFIG.get('w_spatial', 0) > 0):
        try:
            _dsp, _nn = predict_spatial_dip(h, self_name=h.attrs.get('well'))
            arr['dip_spatial'] = _dsp
            _L = CONFIG.get('spatial_conf_L', 5000.0)
            arr['spatial_conf'] = (float(np.exp(-max(_nn, 0.0) / _L))
                                   if (_dsp is not None and _nn is not None) else 0.0)
            diag['dip_spatial'] = round(_dsp, 4) if _dsp is not None else -9
            diag['nn_dist'] = round(_nn, 0) if _nn is not None else -1
            diag['sp_conf'] = round(arr['spatial_conf'], 3)
        except Exception:
            arr['dip_spatial'] = None
            arr['spatial_conf'] = 0.0
    known = int((~arr['blind']).sum())
    diag.update(n=arr['n'], blind_len=int(arr['blind'].sum()), known_len=known,
                dmd=round(arr['dmd'], 3),
                gr_cov=round(float(np.isfinite(arr['gr_raw']).mean()), 3))
    if detect_z_sign(arr) < 0:
        arr = dict(arr, z=-arr['z'])
        status_ok = 'ok_zflip'
    else:
        status_ok = 'ok'
    try:
        ref = build_reference(arr, t, grid_step=CONFIG.get('ref_grid_step', 0.5))   # canonical resolution
        diag['corr'] = round(float(ref[2]), 3)
        tw = t.dropna(subset=['TVT'])
        diag['anchor_margin'] = round(float(min(arr['anchor_tvt'] - tw['TVT'].min(),
                                                tw['TVT'].max() - arr['anchor_tvt'])), 1)
        if ref[2] < min_corr:
            return const, 'fallback_lowcorr', diag
        p1, tube = _ensemble_pred(arr, t, ref)
        diag['drift_span'] = round(float(tube[1] - tube[0] - 2 * CONFIG['tube_margin']), 1)
        # EM refinement: extend the pseudo-typewell with pass-1 blind pairs
        # (damped weight), re-track, average. Guard against divergence.
        arr2 = arr
        if CONFIG['recal']:
            gr_c, gr_raw_c = rolling_affine_correction(
                arr, ref, p1, CONFIG['recal_window_ft'], CONFIG['recal_damp'])
            arr2 = dict(arr, gr=gr_c, gr_raw=gr_raw_c)
        ok = np.isfinite(arr2['gr_raw']) & arr['blind']
        pred = p1
        if ok.sum() > CONFIG['em_min_pairs']:
            ref2 = build_reference(arr2, t, grid_step=CONFIG.get('ref_grid_step', 0.5), extra_tvt=p1[ok],
                                   extra_gr=arr2['gr_raw'][ok], extra_w=EM_WEIGHT)
            p2, _ = _ensemble_pred(arr2, t, ref2, tube=tube)
            diag['em_div'] = round(float(np.abs((p2 - p1)[arr['blind']]).mean()), 2)
            if diag['em_div'] <= EM_DIV_GUARD:
                pred = 0.5 * (p1 + p2)
                if CONFIG.get('em_iters', 1) >= 2:
                    ok2 = (np.isfinite(arr2['gr_raw']) & arr['blind']
                           & np.isfinite(p2))
                    if ok2.sum() > CONFIG['em_min_pairs']:
                        ref3 = build_reference(
                            arr2, t, grid_step=CONFIG.get('ref_grid_step', 0.5), extra_tvt=p2[ok2],
                            extra_gr=arr2['gr_raw'][ok2],
                            extra_w=EM_WEIGHT * 0.5)
                        p3, _ = _ensemble_pred(arr2, t, ref3, tube=tube)
                        if np.abs((p3 - p2)[arr['blind']]).mean() <= EM_DIV_GUARD:
                            pred = (p1 + p2 + p3) / 3.0
        # drift-cancelling branch: robust to slow GR calibration drift along the
        # lateral (invisible to known-zone diagnostics); mixed at fixed weight.
        try:
          if CONFIG['w_dc'] > 0:
            core_dc = build_core(arr2, t, u_window=tube, _ref_cache=ref,
                                 dc_window_ft=CONFIG['dc_window_ft'],
                                 dc_mode=CONFIG['dc_mode'])
            kind, ekw = CONFIG['dc_member']
            E = derive_emissions(core_dc, **ekw)
            Eb, nb, block = block_reduce(core_dc, E)
            us = (solve_viterbi if kind == 'vit' else solve_posterior)(
                core_dc, Eb, nb, block, **SOLVE)
            p_dc = _u_path_to_pred(core_dc, us, nb, block)
            p_dc = _maybe_graze_redo(p_dc, arr2, t, ref, CONFIG['dc_window_ft'], tube)
            m_dc = np.isfinite(p_dc[arr['blind']]).all()
            diag['dc_div'] = round(float(np.abs((p_dc - pred)[arr['blind']]).mean()), 2) if m_dc else -1.0
            if m_dc:
                arr.setdefault('_branch_paths', []).append(p_dc.copy())
                pred = (1 - CONFIG['w_dc']) * pred + CONFIG['w_dc'] * p_dc
        except Exception:
            diag['dc_div'] = -1.0
        # typewell-only branch: reference without the known-zone pseudo-typewell
        try:
          if CONFIG['w_tw'] > 0:
            ref_tw = build_reference(arr, t, grid_step=CONFIG.get('ref_grid_step', 0.5), n0=1e9)
            core_tw = build_core(arr2, t, u_window=tube, _ref_cache=ref_tw)
            E = derive_emissions(core_tw, emis_clip=40.0, rho=0.02)
            Eb, nb, block = block_reduce(core_tw, E)
            us = solve_viterbi(core_tw, Eb, nb, block, **SOLVE)
            p_tw = _u_path_to_pred(core_tw, us, nb, block)
            p_tw = _maybe_graze_redo(p_tw, arr2, t, ref_tw, 0.0, tube)
            if np.isfinite(p_tw[arr['blind']]).all():
                arr.setdefault('_branch_paths', []).append(p_tw.copy())
                pred = (1 - CONFIG['w_tw']) * pred + CONFIG['w_tw'] * p_tw
        except Exception:
            diag['tw_div'] = -1.0
        try:
          if CONFIG['w_dctw'] > 0:
            ref_tw2 = build_reference(arr, t, grid_step=CONFIG.get('ref_grid_step', 0.5), n0=1e9)
            core_x = build_core(arr2, t, u_window=tube, _ref_cache=ref_tw2,
                                dc_window_ft=CONFIG['dc_window_ft'],
                                dc_mode=CONFIG['dc_mode'])
            E = derive_emissions(core_x, emis_clip=40.0, rho=0.02)
            Eb, nb, block = block_reduce(core_x, E)
            us = solve_viterbi(core_x, Eb, nb, block, **SOLVE)
            p_x = _u_path_to_pred(core_x, us, nb, block)
            p_x = _maybe_graze_redo(p_x, arr2, t, ref_tw2, CONFIG['dc_window_ft'], tube)
            if np.isfinite(p_x[arr['blind']]).all():
                arr.setdefault('_branch_paths', []).append(p_x.copy())
                pred = (1 - CONFIG['w_dctw']) * pred + CONFIG['w_dctw'] * p_x
        except Exception:
            diag['dctw_div'] = -1.0
        arr.setdefault('_branch_paths', []).append(pred.copy())
        # spatial branch: tracks the spatially-predicted sloped structural path
        try:
          if CONFIG.get('w_spatial', 0) > 0 and arr.get('dip_spatial') is not None:
            core_sp = build_core(arr2, t, u_window=tube, _ref_cache=ref,
                                 spatial_center=True)
            E = derive_emissions(core_sp, emis_clip=40.0,
                                 rho=CONFIG.get('spatial_rho', 0.05))
            Eb, nb, block = block_reduce(core_sp, E)
            us = solve_viterbi(core_sp, Eb, nb, block, **SOLVE)
            p_sp = _u_path_to_pred(core_sp, us, nb, block)
            p_sp = _maybe_graze_redo(p_sp, arr2, t, ref, 0.0, tube)
            if np.isfinite(p_sp[arr['blind']]).all():
                diag['sp_div'] = round(
                    float(np.abs((p_sp - pred)[arr['blind']]).mean()), 2)
                w_eff = CONFIG['w_spatial'] * arr.get('spatial_conf', 1.0)
                arr['_branch_paths'].append(p_sp.copy())
                pred = (1 - w_eff) * pred + w_eff * p_sp
        except Exception:
            diag['sp_div'] = -1.0
        # learned-emission branch: discriminatively trained matchedness score
        try:
          if CONFIG.get('w_le', 0) > 0:
            core_le = build_core(arr2, t, u_window=tube, _ref_cache=ref)
            E = (np.minimum(core_le['R'], np.float32(25.0)) + learned_emission(core_le)
                 + np.float32(0.02) * core_le['prior_dev'])
            Eb, nb, block = block_reduce(core_le, E)
            us = solve_viterbi(core_le, Eb, nb, block, **SOLVE)
            p_le = _u_path_to_pred(core_le, us, nb, block)
            p_le = _maybe_graze_redo(p_le, arr2, t, ref, 0.0, tube)
            if np.isfinite(p_le[arr['blind']]).all():
                diag['le_div'] = round(
                    float(np.abs((p_le - pred)[arr['blind']]).mean()), 2)
                pred = (1 - CONFIG['w_le']) * pred + CONFIG['w_le'] * p_le
        except Exception:
            diag['le_div'] = -1.0
    except GuardError as g:
        return const, f'fallback_{g.status}', diag
    except Exception as e:
        return const, f'fallback_bug_{type(e).__name__}', diag
    blind = arr['blind']
    out = const.copy()
    dev = pred[blind] - const[blind]
    dev = np.where(np.isfinite(dev), np.clip(dev, -dev_clip, dev_clip), 0.0)
    out[blind] = const[blind] + dev
    try:
        out2 = field_blend(h, out, diag, arr=arr)
        if np.isfinite(out2[blind]).all():
            out = out2
    except Exception:
        pass
    diag['pred_dev'] = round(float(np.abs(out[blind] - const[blind]).mean()), 2)
    return out, status_ok, diag

# ------------------------------------------------------------- validation

def evaluate(wlist, predictor=None, **kw):
    maes, rmses = [], []
    for w in wlist:
        h, t = load_well('train', w)
        pred = (predictor(h, t, **kw) if predictor is not None
                else predict_well(h, t)[0])
        blind = h['TVT_input'].isna().values
        err = pred[blind] - h['TVT'].values[blind]
        maes.append(np.abs(err).mean())
        rmses.append(np.sqrt((err ** 2).mean()))
    return np.mean(maes), np.mean(rmses), np.array(maes)


def summarize_diagnostics(diags):
    """Print aggregate percentiles of per-well diagnostics (population fingerprint)."""
    import collections
    keys = sorted({k for d in diags for k in d})
    print('=== population diagnostics (%d wells) ===' % len(diags))
    for k in keys:
        v = np.array([d[k] for d in diags if k in d], dtype=float)
        if len(v):
            q = np.percentile(v, [10, 50, 90])
            print('%-14s n=%-4d p10=%-9.3g p50=%-9.3g p90=%-9.3g mean=%.3g'
                  % (k, len(v), q[0], q[1], q[2], v.mean()))


## Run-time attestation

In [ ]:
def _attest(namespace):
    import inspect, hashlib
    def norm(s):
        lines = [l.rstrip() for l in s.split(chr(10))]
        while lines and not lines[0]: lines.pop(0)
        while lines and not lines[-1]: lines.pop()
        return chr(10).join(lines)
    parts = [repr(sorted(namespace['CONFIG'].items()))]
    missing = []
    for fn in ['predict_well_diag', 'build_core', 'build_ufield', 'field_blend', '_field_query', 'build_reference', 'build_spatial_map', 'predict_spatial_dip']:
        f = namespace.get(fn)
        if f is None:
            missing.append(fn); continue
        try:
            parts.append(norm(inspect.getsource(f)))
        except Exception as e:
            missing.append(fn + ':' + type(e).__name__)
    h = hashlib.sha256(chr(10).join(parts).encode()).hexdigest()
    return h, missing

MODEL_SHA_EXPECTED = '26dca25a8129a0b14a28a903835b3c7bd00460bc0f572d6384f85e3212466a69'
_h, _missing = _attest(globals())
print('MODEL_SHA expected:', MODEL_SHA_EXPECTED)
print('MODEL_SHA actual:  ', _h)
print('missing sources:', _missing if _missing else 'none')
print('CONFIG sentinels: w_dc=%s w_tw=%s ref_grid_step=%s field_aug=%s field_ivar=%s field_prior=%s'
      % (CONFIG.get('w_dc'), CONFIG.get('w_tw'), CONFIG.get('ref_grid_step'),
         CONFIG.get('field_aug'), CONFIG.get('field_ivar'), CONFIG.get('field_prior')))
print('ATTESTATION:', 'MATCH — verified code is running' if _h == MODEL_SHA_EXPECTED and not _missing
      else '*** MISMATCH — do not trust this run ***')


## Spatial map + structural field

In [ ]:
t0 = time.time()
M = build_spatial_map()
F = build_ufield()
print('spatial map: %d wells | structural field: %s points (%.0fs)'
      % (len(M['dip']), 'none' if F is None else F['n'], time.time() - t0))

## Inference + diagnostics

## Test inference / submission — **guarded off**

Skipped in this fork. `RUN_TEST_INFERENCE` stays `False`.

In [ ]:
RUN_TEST_INFERENCE = False   # OOF fork: do not write a submission from patched code
if RUN_TEST_INFERENCE:
    rows = []
    t0 = time.time()
    wl = wells('test')
    print(len(wl), 'test wells')
    status_counts = {}
    diags = []
    for i, w in enumerate(wl):
        h, t = load_well('test', w)
        pred, status, d = predict_well_diag(h, t)
        status_counts[status] = status_counts.get(status, 0) + 1
        diags.append(d)
        for j in np.where(h['TVT_input'].isna().values)[0]:
            rows.append((f'{w}_{j}', pred[j]))
        if (i + 1) % 25 == 0 or i == len(wl) - 1:
            print(f'{i+1}/{len(wl)}  ({time.time()-t0:.0f}s)', flush=True)
    print('status counts:', status_counts)
    summarize_diagnostics(diags)
    sub = pd.DataFrame(rows, columns=['id', 'tvt'])
    assert sub.tvt.notna().all(), 'non-finite predictions escaped the guards'
    sub.to_csv('submission.csv', index=False)
    print('wrote submission.csv:', len(sub), 'rows')

---
# P0 – P6 — OOF instrumentation

Run these in order after everything above. **P5 must pass before P6.**

In [ ]:
# ===== P0: preflight =====
# Asserts the v22 globals this patch depends on, checks field_aug, and snapshots
# source hashes so the patch is auditable. Nothing is modified here.
import hashlib, inspect, sys
import numpy as np

_REQUIRED = ['build_ufield', '_field_query', 'field_blend',
             'predict_well_diag', 'load_well', 'wells', 'CONFIG']
_missing = [n for n in _REQUIRED if n not in globals()]
if _missing:
    raise NameError(
        'v22 globals not found: %s\n'
        'Run your v22 notebook top-to-bottom FIRST (model-code cell + the '
        'spatial-map/field cell), then run these patch cells.' % _missing)

# --- field_aug guard --------------------------------------------------------
# P2 drops the field_aug branch from the query path as dead code. That is only
# safe while the flag is off. Fail loudly rather than diverge silently.
if CONFIG.get('field_aug', False):
    raise RuntimeError(
        'CONFIG["field_aug"] is True. The patched _field_query in P2 omits the '
        'field_aug branch (LB-falsified, assumed off). Either set it False for '
        'this analysis run, or restore that branch in P2 before proceeding.')
print('field_aug guard OK (False)')

def _srchash(fn):
    try:
        return hashlib.sha256(inspect.getsource(fn).encode()).hexdigest()[:16]
    except Exception:
        return '<unavailable>'

_PRE = {n: _srchash(globals()[n]) for n in _REQUIRED if callable(globals()[n])}
print('\nPREFLIGHT OK — all v22 globals present')
for k, v in _PRE.items():
    print('  %-20s src sha256[:16] = %s' % (k, v))

# --- does v22 already thread self_well into _field_query? --------------------
# If it does, the _CUR_WELL wrapper in P3 is redundant belt-and-braces (harmless:
# an explicit self_well argument always wins). If it does not, _CUR_WELL is the
# only thing making exclusion fire. Either way P5 is the real arbiter.
try:
    _pwd_src = inspect.getsource(predict_well_diag)
    _n_pass = _pwd_src.count('self_well=')
    _n_call = _pwd_src.count('_field_query(')
    print('\n_field_query call sites in predict_well_diag : %d' % _n_call)
    print('  ...of which pass self_well explicitly       : %d' % _n_pass)
    if _n_call > _n_pass:
        print('  -> %d call site(s) pass NOTHING. For those, the _CUR_WELL plumbing'
              % (_n_call - _n_pass))
        print('     in P3 is the only thing making exclusion fire. P5 confirms.')
except Exception:
    print('\ncould not inspect predict_well_diag source (not fatal)')

# v22 already computes mask_self = (wid[idx] == self_well), but build_ufield sets
# wid = ['']*n_train, so mask_self is always False and the machinery is inert.
# Note also that it masks AFTER the k-NN query: fixing wid alone would zero all
# 40 neighbours on a well's own lateral and silently disable the field blend.
# P1 populates wid; P2 replaces the mask with an exclude-then-build rebuild.
try:
    _bu = inspect.getsource(build_ufield)
    print('\nbuild_ufield currently sets wid to empty strings:',
          "wid = [''] * n_train" in _bu)
except Exception:
    pass

_PATCH_APPLIED = globals().get('_PATCH_APPLIED', set())
print('\npatches already applied in this session:', sorted(_PATCH_APPLIED) or 'none')


## P1 — tag each field point with its well

`build_ufield` builds `wid = [''] * n_train`, so there is no way to tell which
field point came from which well and self-exclusion is impossible. This rewrites
four anchors in the function's **source text**, leaving the datum/`offs` head
untouched:

- `pts = []; us = []` → also open a `widp` list
- the `pts.append(...)` line → tag every point with its well id
- `wid = [''] * n_train` → `wid = list(widp)`
- after `P = np.vstack(pts)` → stash the raw point array for P2's tree rebuilds

Every replacement asserts `count == 1`, and the patched source is `ast.parse`d
before it's exec'd. The full patched source is printed so you can eyeball the
diff. If an anchor misses, the cell prints your real source and stops having
modified nothing — send me that dump and the anchors get re-cut in two minutes.

In [ ]:
# ===== P1: patch build_ufield source (verified, exactly-once replacements) =====
import inspect, hashlib, textwrap, ast

_src0 = inspect.getsource(build_ufield)
print('build_ufield BEFORE sha256 =', hashlib.sha256(_src0.encode()).hexdigest())

_src = textwrap.dedent(_src0)

_ANCHORS = [
    ('A1', "pts = []; us = []",
           "pts = []; us = []; widp = []"),
    ('A2', "pts.append(np.column_stack([h.X.values[m][::8], h.Y.values[m][::8]]))",
           "_p = np.column_stack([h.X.values[m][::8], h.Y.values[m][::8]])\n"
           "        pts.append(_p)\n"
           "        widp.extend([w] * len(_p))"),
    ('A3', "wid = [''] * n_train",
           "wid = list(widp)"),
    ('A4', "P = np.vstack(pts); Uv = np.concatenate(us)",
           "P = np.vstack(pts); Uv = np.concatenate(us)\n"
           "    globals()['_UFIELD_PTS'] = P"),
]

_FAILED = []
for _tag, _old, _new in _ANCHORS:
    _n = _src.count(_old)
    print('  anchor %s: %d match(es)' % (_tag, _n))
    if _n != 1:
        _FAILED.append((_tag, _old, _n))

if _FAILED:
    print('\n' + '=' * 72)
    print('ANCHOR MISMATCH — patch NOT applied, nothing was modified.')
    print('Your v22 source differs from what this patch expects. Full source')
    print('below; send it over and the anchors get re-cut against your text.')
    print('=' * 72)
    for _tag, _old, _n in _FAILED:
        print('  %s expected 1 match, found %d, for:\n    %r' % (_tag, _n, _old))
    print('-' * 72)
    print(_src0)
    raise SystemExit('P1 aborted: anchor mismatch')

for _tag, _old, _new in _ANCHORS:
    _src = _src.replace(_old, _new)

ast.parse(_src)                      # syntax gate before exec
exec(compile(_src, '<build_ufield_patched>', 'exec'), globals())

# Register the patched source in linecache. exec'd code has no file on disk, so
# inspect.getsource() raises OSError -- which would also break v22's own
# _attest(), since that calls getsource on build_ufield.
import linecache
linecache.cache['<build_ufield_patched>'] = (
    len(_src), None, _src.splitlines(True), '<build_ufield_patched>')

# Record success IMMEDIATELY. Everything below is reporting, and a cosmetic
# failure there must not leave the patch applied but unregistered -- P6 asserts
# on _PATCH_APPLIED.
assert 'widp' in build_ufield.__code__.co_varnames, 'exec did not rebind build_ufield'
_PATCH_APPLIED.add('P1'); globals()['_PATCH_APPLIED'] = _PATCH_APPLIED
print('PATCH P1 APPLIED - field points now carry per-point well ids')

try:
    print('build_ufield AFTER  sha256 =',
          hashlib.sha256(inspect.getsource(build_ufield).encode()).hexdigest())
except OSError as _e:
    print('(AFTER hash unavailable: %s -- cosmetic only, patch IS applied)' % _e)
print()
print('--- patched source (eyeball the four edits) ---')
print(_src)


## P2 — exclude-then-build self-exclusion

This is the subtle one, and it's why the obvious fix fails.

A well sits *on* its own dense lateral. With `field_sub=8` a well contributes
roughly 820 field points spaced ~8 ft apart, spanning ±160 ft around any station
on it. So for any query point on that lateral, **all 40 nearest neighbours are the
well's own points** — the next well is hundreds of feet away.

Masking those neighbours *after* the k-nearest query therefore zeroes every weight
→ `s ≈ 0` → `est = NaN` → the `okk.sum() < 50` guard trips → `field_blend` returns
`pred` untouched. You wouldn't get a leak. You'd get **v22 with the field blend
silently switched off**, scoring maybe 7.5–8 instead of ~6.55, and the join would
compare the BiGRU against a crippled v22.

The fix is to rebuild the KD-tree *without* the well's points, then query. The
current well arrives via the module-global `_CUR_WELL` (set by the P3 wrapper), so
no call site changes — and an explicit `self_well` argument always wins, so this
is correct whether or not v22 already threads it through.

The point array is read from `_UFIELD['pts']` when present, falling back to the
`_UFIELD_PTS` global that P1 sets. The tree cache holds exactly one entry: each
tree spans the full field (~600k points), so caching several would cost hundreds
of MB for no benefit when wells are processed one at a time.

In [ ]:
# ===== P2: exclude-then-build field query =====
from scipy.spatial import cKDTree

_CUR_WELL = globals().get('_CUR_WELL', None)   # set by the P3 wrapper
_EXCL_CACHE = {}

def _field_points():
    """Raw field point array: _UFIELD['pts'] if present, else the P1 global."""
    F = _UFIELD
    pts = F.get('pts') if isinstance(F, dict) else None
    if pts is None:
        pts = globals().get('_UFIELD_PTS')
    if pts is None:
        raise RuntimeError('field point array missing — re-run P1 then P4')
    return np.asarray(pts)


def _excluded_field(self_well):
    """Return (tree, U) with self_well's own points removed.

    Masking neighbours AFTER a shared-tree query is wrong: a well sits on its own
    dense lateral, so all k neighbours are its own points and masking zeroes every
    weight, silently disabling the blend. Rebuild without them instead.
    """
    F = _UFIELD
    if self_well is None:
        return F['tree'], F['U']
    hit = _EXCL_CACHE.get(self_well)
    if hit is not None:
        return hit
    keep = np.asarray(F['wid']) != self_well
    if keep.all():                      # test wells aren't in the field: no-op
        out = (F['tree'], F['U'])
    else:
        out = (cKDTree(_field_points()[keep]), np.asarray(F['U'])[keep])
    _EXCL_CACHE.clear()                 # only the current well is ever needed
    _EXCL_CACHE[self_well] = out
    return out


def _field_query(xq, yq, self_well=None):
    F = _UFIELD
    k = int(CONFIG.get('field_k', 40))
    soft = float(CONFIG.get('field_soft', 400.0))
    if self_well is None:               # explicit argument always wins
        self_well = globals().get('_CUR_WELL', None)
    tree, U = _excluded_field(self_well)
    dd, idx = tree.query(np.column_stack([xq, yq]), k=k)
    wgt = 1.0 / (dd + soft) ** 2
    s = wgt.sum(1)
    est = np.einsum('nk,nk->n', wgt, U[idx]) / np.maximum(s, 1e-12)
    var = np.einsum('nk,nk->n', wgt,
                    (U[idx] - est[:, None]) ** 2) / np.maximum(s, 1e-12)
    est[s <= 1e-12] = np.nan
    return est, dd[:, 0], np.sqrt(np.maximum(var, 0))

print('PATCH P2 APPLIED — _field_query excludes-then-builds, driven by _CUR_WELL')
_PATCH_APPLIED.add('P2'); globals()['_PATCH_APPLIED'] = _PATCH_APPLIED


## P3 — branch spread + current-well plumbing

Two wrappers, both idempotent (guarded on `_orig_*` already existing, so
re-running cannot double-wrap).

**`field_blend`** — v22 already builds `arr['_branch_paths']`, the per-branch
predictions whose disagreement drives the inverse-variance fusion. That spread
*is* the candidate routing signal. The wrapper records the mean over blind
stations of the median-absolute-deviation across branch paths into
`diag['branch_spread_mean']`, then defers to the original.

The blind mask is resolved defensively: `arr['blind']` first (v22's canonical
definition, and the branch paths live in `arr` space), falling back to
`h.TVT_input.isna()` only if the lengths say `arr` space and `h` space agree. If
neither lines up, the diagnostic is skipped rather than computed on a misaligned
mask — a wrong routing variable is worse than a missing one.

**`predict_well_diag`** — sets `_CUR_WELL` from `h.attrs['well']` for the duration
of the call, in `try/finally` so it's always restored, including on exceptions.

In [ ]:
# ===== P3: field_blend spread capture + predict_well_diag well plumbing =====

def _resolve_blind(h, arr, n):
    """Blind mask of length n, or None if no aligned mask can be found."""
    b = arr.get('blind') if isinstance(arr, dict) else None
    if b is not None and len(b) == n:
        return np.asarray(b, dtype=bool)
    try:
        hb = ~np.isfinite(h['TVT_input'].values.astype(float))
        if len(hb) == n:
            return hb
    except Exception:
        pass
    return None


if '_orig_field_blend' not in globals():
    _orig_field_blend = field_blend

def field_blend(*a, **k):
    """Record per-well branch spread into diag, then defer to v22's original."""
    h = k.get('h', a[0] if len(a) >= 1 else None)
    diag = k.get('diag', a[2] if len(a) >= 3 else None)
    arr = k.get('arr', a[3] if len(a) >= 4 else None)
    try:
        bp_list = arr.get('_branch_paths') if isinstance(arr, dict) else None
        if isinstance(diag, dict) and bp_list is not None and len(bp_list) >= 2:
            bp = np.stack(bp_list)
            sp = np.median(np.abs(bp - np.median(bp, axis=0)), axis=0)
            blind = _resolve_blind(h, arr, len(sp))
            if blind is not None:
                v = sp[blind]
                v = v[np.isfinite(v)]
                if len(v):
                    diag['branch_spread_mean'] = round(float(v.mean()), 3)
                    diag['branch_spread_n'] = int(len(v))
    except Exception:
        pass                              # a diagnostic must never break a prediction
    return _orig_field_blend(*a, **k)


if '_orig_predict_well_diag' not in globals():
    _orig_predict_well_diag = predict_well_diag

def predict_well_diag(h, t, *a, **k):
    """Publish the current well to _CUR_WELL so _field_query can self-exclude."""
    global _CUR_WELL
    _prev = globals().get('_CUR_WELL', None)
    try:
        _CUR_WELL = h.attrs.get('well') if hasattr(h, 'attrs') else None
    except Exception:
        _CUR_WELL = None
    try:
        return _orig_predict_well_diag(h, t, *a, **k)
    finally:
        _CUR_WELL = _prev

print('PATCH P3 APPLIED — branch_spread_mean captured; _CUR_WELL plumbed')
print('  wrapped field_blend       :', _orig_field_blend)
print('  wrapped predict_well_diag :', _orig_predict_well_diag)
_PATCH_APPLIED.add('P3'); globals()['_PATCH_APPLIED'] = _PATCH_APPLIED


## P4 — rebuild the field

The field must be rebuilt so `_UFIELD['wid']` picks up the per-point well tags and
the point array gets populated. Without this, P2 has nothing to exclude against.

In [ ]:
# ===== P4: rebuild the structural field with tagged well ids =====
import time
assert not CONFIG.get('field_aug', False), 'field_aug flipped True since P0'

_t0 = time.time()
_UFIELD = None
F = build_ufield()
print('field rebuilt in %.0fs' % (time.time() - _t0))

_wid = np.asarray(_UFIELD['wid'])
_n_unique = len(np.unique(_wid))
_pts = _field_points()
print('field points      : %d' % len(_UFIELD['U']))
print('unique well tags  : %d' % _n_unique)
print('point array shape : %s  (source: %s)'
      % (_pts.shape, 'ufield["pts"]' if _UFIELD.get('pts') is not None
                     else '_UFIELD_PTS global'))

assert _n_unique > 100, (
    'wid tagging did not take effect (%d unique tags). Re-run P1, then P4.' % _n_unique)
assert len(_pts) == len(_UFIELD['U']) == len(_wid), 'field arrays misaligned'
_EXCL_CACHE.clear()
print('\nP4 OK — field carries per-point well ids')


## P5 — validation gate

This decides whether anything below is worth reading. It runs the field query on
one training well twice — self-exclusion off, then on — and reports nearest-
neighbour distance and field-only blind MAE for each.

| | nn_dist (median) | field-only blind MAE |
|---|---|---|
| self-**included** (leaked) | ~2 ft | ~0.2–0.4 |
| self-**excluded** (real) | hundreds of ft | ~13–15 |

The leaked MAE being near zero is the point: the well is reading its own answer
off its own lateral. If the two rows look alike, exclusion isn't firing.

The third check matters as much: the excluded estimate must stay **finite**.
All-NaN is the tree-rebuilt-empty failure that silently switches the blend off.

In [ ]:
# ===== P5: VALIDATION GATE — prove self-exclusion is firing =====
_tw = wells('train')
_w = _tw[100] if len(_tw) > 100 else _tw[0]
_h, _t = load_well('train', _w)
_h.attrs['well'] = _w

_z = _h['Z'].values.astype(float)
_tvt = _h['TVT'].values.astype(float)
_tin = _h['TVT_input'].values.astype(float)
_blind = ~np.isfinite(_tin)
_bok = _blind & np.isfinite(_tvt)
_ku = (~_blind) & np.isfinite(_tin)

print('gate well: %s   blind stations: %d   known: %d\n' % (_w, _bok.sum(), _ku.sum()))
print('%-16s %12s %12s %10s' % ('mode', 'nn_dist(med)', 'blind MAE', 'finite%'))
print('-' * 54)

_res = {}
for _tag, _sw in [('self-included', None), ('self-excluded', _w)]:
    _EXCL_CACHE.clear()
    _est, _dmin, _ = _field_query(_h.X.values, _h.Y.values, self_well=_sw)
    _fin = np.isfinite(_est)
    _ok = _ku & _fin
    if _ok.sum() < 10:
        print('%-16s %12s %12s %9.1f%%' % (_tag, 'n/a', 'NO ANCHOR', 100 * _fin.mean()))
        _res[_tag] = (np.nan, np.nan); continue
    _off = np.median((_tin + _z)[_ok] - _est[_ok])
    _ftvt = _est + _off - _z
    _m = _bok & np.isfinite(_ftvt)
    _res[_tag] = (float(np.median(_dmin[_bok])),
                  float(np.abs(_ftvt[_m] - _tvt[_m]).mean()))
    print('%-16s %12.0f %12.2f %9.1f%%' % (_tag, _res[_tag][0], _res[_tag][1],
                                           100 * _fin.mean()))

_EXCL_CACHE.clear()
_nn_in, _mae_in = _res['self-included']
_nn_ex, _mae_ex = _res['self-excluded']

print('\n' + '=' * 54)
_pass = True
if not np.isfinite(_mae_ex):
    print('FAIL: excluded query produced no usable anchor (all-NaN field).')
    print('      This is the "field blend silently off" failure mode.')
    _pass = False
if np.isfinite(_nn_ex) and np.isfinite(_nn_in) and _nn_ex < 10 * max(_nn_in, 1.0):
    print('FAIL: nn_dist barely moved (%.0f -> %.0f). Exclusion is not firing.'
          % (_nn_in, _nn_ex))
    _pass = False
if np.isfinite(_mae_ex) and _mae_ex < 3.0:
    print('FAIL: excluded blind MAE %.2f implausibly low — a leak survived.' % _mae_ex)
    _pass = False
if _pass:
    print('GATE PASSED — self-exclusion is real.')
    print('  leaked MAE %.2f (nn %.0f ft)  ->  honest MAE %.2f (nn %.0f ft)'
          % (_mae_in, _nn_in, _mae_ex, _nn_ex))
    print('  Honest MAE should sit near 13-15. Proceed to P6.')
else:
    print('\nDO NOT RUN P6 until this passes.')
print('=' * 54)


## P6 — the OOF loop

Scores v22 over the training wells with each well excluded from its own field,
mirroring deployment exactly: a test well is never in the field.

`v22_oof.pkl` carries six things, keyed by well:

- `err` — per-well blind-zone MAE in **absolute TVT space**
- `pred` — blind station indices and predicted TVT, for the join and any offline blend
- `spread` — `branch_spread_mean`, routing candidate #1
- `nn` — `nn_dist`, field-support distance, routing candidate #2
- `field_conf` — routing candidate #3
- `status` — which code path each well took, so failures can be clustered

**The comparability trap.** The BiGRU trains on the increment target with its own
per-well datum and `u_last`; v22 has its own anchor logic. They're only comparable
if both errors are `abs(pred_TVT − true_TVT)` over the *identical* blind mask of
the *identical* wells. This loop is in absolute TVT space for exactly that reason
— make sure the BiGRU side is too, or you'll chase a phantom.

**Validity gate.** Final OOF MAE should land near **~6.5**. Around 7.5–8 means the
field blend is off. Well below 6 means a leak survived.

Budget 45–75 min — roughly 3.6 s/well plus one tree rebuild per well.
Checkpoints every 25 wells, so a timeout doesn't cost the run.

In [ ]:
# ===== P6: v22 OOF over the training wells (leak-free via self-exclusion) =====
import pickle, time, traceback
from collections import Counter

assert {'P1', 'P2', 'P3'} <= _PATCH_APPLIED, \
    'run P1-P3 first (applied: %s)' % sorted(_PATCH_APPLIED)

tr_wells = wells('train')
v22_err, v22_pred, v22_spread, v22_nn = {}, {}, {}, {}
v22_fc, v22_status = {}, {}
skipped = []
t0 = time.time()

def _save(partial):
    pickle.dump({'err': v22_err, 'pred': v22_pred, 'spread': v22_spread,
                 'nn': v22_nn, 'field_conf': v22_fc, 'status': v22_status,
                 'skipped': skipped, 'partial': partial},
                open('v22_oof.pkl', 'wb'))

for i, w in enumerate(tr_wells):
    try:
        h, t = load_well('train', w)
        h.attrs['well'] = w                    # -> _CUR_WELL -> self-exclusion active

        truth = h['TVT'].values.astype(float)
        blind = ~np.isfinite(h['TVT_input'].values.astype(float))
        m = blind & np.isfinite(truth)
        if m.sum() < 1:
            skipped.append((w, 'no scorable blind stations'))
            v22_status[w] = 'skip_no_blind'
            continue

        pred, status, diag = predict_well_diag(h, t)
        pred = np.asarray(pred, dtype=float)

        good = m & np.isfinite(pred)
        if good.sum() < 1:
            skipped.append((w, 'all predictions NaN'))
            v22_status[w] = 'skip_all_nan'
            continue

        v22_err[w] = float(np.abs(pred[good] - truth[good]).mean())
        v22_pred[w] = dict(blind_idx=np.where(good)[0].astype(np.int32),
                           tvt_hat=pred[good].astype(np.float32))
        v22_spread[w] = float(diag.get('branch_spread_mean', np.nan))
        v22_nn[w] = float(diag.get('nn_dist', np.nan))
        v22_fc[w] = float(diag.get('field_conf', np.nan))
        v22_status[w] = str(status)

    except Exception as e:
        skipped.append((w, repr(e)[:120]))
        v22_status[w] = 'error_' + type(e).__name__
        if len(skipped) <= 3:
            traceback.print_exc()

    if (i + 1) % 25 == 0:
        _run = np.mean(list(v22_err.values())) if v22_err else np.nan
        _el = time.time() - t0
        print('%4d/%d  running mean err %.3f  skipped %d  [%.0fs, ~%.0fs left]'
              % (i + 1, len(tr_wells), _run, len(skipped), _el,
                 _el / (i + 1) * (len(tr_wells) - i - 1)), flush=True)
        _save(True)

_save(False)

_errs = np.array(list(v22_err.values()))
_oof = float(_errs.mean())
_sp = np.array([v22_spread[w] for w in v22_err])
_nn = np.array([v22_nn[w] for w in v22_err])
_fc = np.array([v22_fc[w] for w in v22_err])

print('\n' + '=' * 66)
print('saved v22_oof.pkl | v22 OOF MAE %.3f over %d wells (%.0f min)'
      % (_oof, len(v22_err), (time.time() - t0) / 60))
print('  per-well err: p10 %.2f  median %.2f  p90 %.2f  max %.2f'
      % tuple(np.percentile(_errs, [10, 50, 90]).tolist() + [_errs.max()]))
print('  branch_spread present : %.1f%%' % (100 * np.isfinite(_sp).mean()))
print('  nn_dist present       : %.1f%%' % (100 * np.isfinite(_nn).mean()))
print('  field_conf present    : %.1f%%' % (100 * np.isfinite(_fc).mean()))
print('  statuses:', Counter(v22_status.values()).most_common(6))
if skipped:
    print('  skipped %d wells; first few: %s' % (len(skipped), skipped[:5]))

print('-' * 66)
if 6.0 <= _oof <= 7.2:
    print('VALIDITY GATE PASSED — %.3f is in range of v22\'s known local ~6.5.' % _oof)
elif _oof > 7.2:
    print('GATE FAILED (%.3f too high) — the field blend is likely switched off.' % _oof)
    print('  Check P5 passed and that P1->P4 ran in order.')
else:
    print('GATE FAILED (%.3f too low) — a leak likely survived. Re-check P5.' % _oof)

if np.isfinite(_fc).mean() < 0.80:
    print('\nNOTE: field_conf present on only %.1f%% of wells. If your v22 diag')
    print('  never had a "field_conf" key this is expected and harmless — the')
    print('  other two routing candidates are unaffected. If it DOES have one,')
    print('  this says the field disengaged on most wells; investigate before')
    print('  trusting the join.')
print('=' * 66)
print('\nDownload v22_oof.pkl from the Output tab, then run the join against')
print('bigru_oof.pkl. Treat anything under ~0.2 local improvement as noise.')


## Notes

**Do not run v22's submission cell in this fork.** `field_blend` and
`predict_well_diag` are wrapped and `build_ufield` is rewritten in memory. The
wrappers are behaviour-preserving on test wells (`_CUR_WELL` is `None`,
`keep.all()` is True, so the query is bit-identical), but there's no reason to
take the risk on a scoring run. Keep the real v22 sealed.

**Re-running is safe.** P1 asserts exactly-once anchors before touching anything.
P2 is a plain redefinition. P3 guards on `_orig_*` so it can't double-wrap. P4
clears `_EXCL_CACHE`.

**On `field_conf`.** It's captured via `diag.get('field_conf', np.nan)`, so if
your v22 doesn't produce that key the value is simply NaN and nothing breaks. The
P6 note distinguishes the two readings of a low presence rate — key absent
(harmless) versus field disengaging (not harmless) — because they look identical
in the output and mean opposite things.

**Reading the join.** Across v19/v22/v23/v24 the local and leaderboard orderings
came out perfectly inverted, with spreads of 0.04 local and 0.02 LB. That's almost
certainly noise rather than genuine anti-correlation, but the implication holds
either way: local differences at the 0.04 scale carry no predictive signal for the
leaderboard. Don't ship on a 0.05.

**One asymmetry, and it favours you.** The BiGRU's OOF is a true 5-fold held-out
number. v22's OOF here benefits from a field built over all training wells minus
self, which is slightly more favourable to v22. So if a blend wins in this
comparison, it should win by at least that much on the leaderboard, not less.